# Práctica 2 de PySpark: ChaskiFood

Segunda práctica de PySpark en Databricks Free Edition. En la primera trabajamos con tablas que ya venían limpias (`samples.nyctaxi`, `samples.tpch`). Ahora toca algo más parecido al trabajo real: llegan archivos crudos a un volumen y hay que convertirlos en tablas confiables para responder preguntas de negocio.

El caso: ChaskiFood es una app de delivery (ficticia) que opera en Lima, Arequipa y Trujillo. El equipo de datos recibe estos archivos:

| Archivo | Formato | Contenido |
|---|---|---|
| `restaurantes.csv` | CSV separado por `;` | restaurantes afiliados y su comisión |
| `clientes.csv` | CSV | clientes registrados, con bastante suciedad |
| `pedidos/pedidos_2025_MM.json` | JSON Lines, un archivo por mes | pedidos de enero a junio de 2025, con datos anidados |
| `actualizaciones/restaurantes_2025_07.csv` | CSV separado por `;` | cambios de restaurantes de julio |

Vamos a armar un pipeline por capas (arquitectura medallón):

```
raw (volumen)           bronze                   silver                     gold
restaurantes.csv  --->  bronze_restaurantes ---> silver_restaurantes --+
clientes.csv      --->  bronze_clientes     ---> silver_clientes     --+--> gold_ventas_diarias
pedidos/*.json    --->  bronze_pedidos      ---> silver_pedidos      --+    gold_segmentos_clientes
```

- bronze: los datos tal como llegan, más columnas de control (archivo de origen, fecha de ingesta)
- silver: datos limpios, sin duplicados y con los tipos correctos
- gold: tablas agregadas, listas para reportes

Se repasa todo lo de la práctica 1 (crear DataFrames, filtros, columnas nuevas, agregaciones, joins, ventanas, SQL, limpieza y escritura) y se sube un poco el nivel con:
- leer CSV y JSON con esquema explícito y detectar registros corruptos
- columnas anidadas (`struct` y `array`), `explode` y funciones de orden superior
- expresiones regulares y fechas que vienen en varios formatos
- `pivot`, `collect_set`, mediana y agregaciones condicionales
- joins `left_semi` y joins por rango
- ventanas con marcos móviles, porcentajes sobre el total y `ntile`
- SQL con CTE y consultas con parámetros
- Delta Lake: `MERGE INTO`, historial y time travel

Igual que antes, debajo de cada ejercicio hay una celda que comprueba el resultado: si imprime "Correcto", sigue; si sale un AssertionError, lee el mensaje y revisa tu código.

Antes de empezar:
1. Conecta el notebook a Serverless (selector de cómputo, arriba a la derecha).
2. Ejecuta las celdas en orden con Shift + Enter, porque casi todos los ejercicios usan resultados de los anteriores.
3. Si te pierdes, puedes volver a ejecutar todo desde el principio: la celda que genera los datos siempre produce exactamente los mismos archivos.

## 0. Preparación
Esta vez creamos un esquema propio, `workspace.chaskifood`, para no mezclar con lo de la práctica 1, y dentro de él un volumen `archivos` donde van a caer los archivos crudos.

Con `setCurrentCatalog` y `setCurrentDatabase` dejamos ese esquema como el actual, así podemos escribir `saveAsTable("silver_pedidos")` o `FROM silver_pedidos` sin poner el nombre completo. Si el notebook se desconecta, vuelve a ejecutar esta celda.

In [0]:
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

CATALOGO = "workspace"
ESQUEMA = "chaskifood"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.archivos")
spark.catalog.setCurrentCatalog(CATALOGO)
spark.catalog.setCurrentDatabase(ESQUEMA)

RUTA_VOLUMEN = f"/Volumes/{CATALOGO}/{ESQUEMA}/archivos"
RUTA_RAW = RUTA_VOLUMEN + "/raw"

def ok(mensaje="Correcto"):
    print(mensaje)

print("Versión de Spark:", spark.version)
print("Esquema actual:", spark.catalog.currentCatalog(), ".", spark.catalog.currentDatabase())

Versión de Spark: 4.2.0
Esquema actual: workspace . chaskifood


### Generar los archivos crudos
La siguiente celda escribe los archivos en `RUTA_RAW`. No hace falta entenderla (es Python normal, sin Spark): simula lo que nos mandarían los sistemas de la empresa, con los errores típicos que trae la data real. Usa una semilla fija, así que todos obtienen exactamente los mismos datos.

In [0]:
# Genera los archivos crudos de ChaskiFood dentro del volumen.
# No hace falta entender esta celda: simula los archivos que nos mandarían los sistemas de la empresa.
# Usa una semilla fija, así que siempre produce exactamente los mismos datos.
import csv, json, math, os, random, unicodedata
from datetime import date, datetime, timedelta


def generar_datos_chaskifood(ruta, semilla=2025):
    rng = random.Random(semilla)
    for carpeta in ("", "/pedidos", "/actualizaciones"):
        os.makedirs(ruta + carpeta, exist_ok=True)

    def sin_tildes(texto):
        return unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()

    def fecha_al_azar(desde, hasta):
        return desde + timedelta(days=rng.randint(0, (hasta - desde).days))

    def poisson(media):
        limite, k, p = math.exp(-media), 0, 1.0
        while True:
            p *= rng.random()
            if p <= limite:
                return k
            k += 1

    def escribir_csv(nombre, filas, sep=","):
        with open(f"{ruta}/{nombre}", "w", encoding="utf-8", newline="") as f:
            csv.writer(f, delimiter=sep).writerows(filas)

    def ensuciar(texto):
        r = rng.random()
        return texto.upper() if r < 0.15 else texto.lower() if r < 0.30 else f"  {texto} " if r < 0.40 else texto

    # ---------------- restaurantes ----------------
    MENUS = {
        "Pollería": [("1/4 de pollo a la brasa", 21.9), ("1/2 pollo a la brasa", 38.9), ("Pollo entero a la brasa", 69.9),
                     ("Porción de papas fritas", 9.9), ("Ensalada clásica", 8.5), ("Chicha morada 1L", 12.0)],
        "Chifa": [("Arroz chaufa", 24.9), ("Tallarín saltado", 26.9), ("Wantán frito", 14.9),
                  ("Chancho al tamarindo", 32.9), ("Kam lu wantán", 34.9), ("Inca Kola 1L", 8.0)],
        "Cevichería": [("Ceviche clásico", 35.9), ("Leche de tigre", 18.9), ("Jalea mixta", 42.9),
                       ("Arroz con mariscos", 38.9), ("Chicharrón de pescado", 33.9)],
        "Pizzería": [("Pizza americana familiar", 45.9), ("Pizza pepperoni familiar", 49.9),
                     ("Pizza hawaiana mediana", 34.9), ("Pan al ajo", 11.9), ("Gaseosa 1.5L", 10.0)],
        "Hamburguesas": [("Hamburguesa clásica", 19.9), ("Hamburguesa royal", 24.9), ("Hamburguesa a lo pobre", 27.9),
                         ("Papas fritas", 9.9), ("Milkshake de lúcuma", 14.9)],
        "Sushi": [("Acevichado roll", 32.9), ("Maki furai", 29.9), ("Nigiri mixto", 36.9), ("Gyozas", 18.9)],
        "Criolla": [("Lomo saltado", 36.9), ("Ají de gallina", 28.9), ("Arroz con pollo", 26.9),
                    ("Causa limeña", 19.9), ("Seco de res", 34.9), ("Chicha morada 1L", 12.0)],
        "Vegetariana": [("Bowl de quinua", 27.9), ("Hamburguesa de lentejas", 24.9), ("Ensalada andina", 22.9),
                        ("Jugo verde", 12.9)],
    }
    NOMBRES_REST = {
        "Pollería": ["Pollos Don Brasa", "La Leña Dorada", "El Gallo Fino"],
        "Chifa": ["Chifa Dragón de Oro", "Chifa Bambú", "Chifa Loto Azul"],
        "Cevichería": ["Mar Picante", "Ola Marina", "Caleta Sabrosa"],
        "Pizzería": ["Pizza Nostra", "Forno Andino", "La Masa Madre"],
        "Hamburguesas": ["Burger Chalaca", "Doble Queso", "La Parrilla Urbana"],
        "Sushi": ["Nikkei Roll", "Kaizen Sushi"],
        "Criolla": ["Sabor Criollo", "El Fogón Limeño", "La Sazón de Mamá Rosa"],
        "Vegetariana": ["Verde Quinua", "Raíz Bowl"],
    }
    DISTRITOS = {
        "Lima": ["Miraflores", "San Isidro", "Surco", "Barranco", "San Borja", "Jesús María", "Los Olivos",
                 "San Miguel", "La Molina"],
        "Arequipa": ["Cercado", "Yanahuara", "Cayma", "Paucarpata"],
        "Trujillo": ["Centro", "Víctor Larco", "La Esperanza"],
    }
    REPARTIDORES = {"Lima": ["R01", "R02", "R03", "R04", "R05", "R06"],
                    "Arequipa": ["R07", "R08", "R09"], "Trujillo": ["R10", "R11", "R12"]}

    categorias = list(MENUS)
    restaurantes = []
    for i, ciudad in enumerate(["Lima"] * 24 + ["Arequipa"] * 9 + ["Trujillo"] * 7, start=1):
        cat = categorias[(i - 1) % len(categorias)]
        restaurantes.append({"id": i, "nombre": rng.choice(NOMBRES_REST[cat]), "categoria": cat, "ciudad": ciudad,
                             "distrito": rng.choice(DISTRITOS[ciudad]), "comision": rng.choice([12, 15, 18, 20, 22.5]),
                             "fecha_alta": fecha_al_azar(date(2019, 1, 1), date(2024, 10, 1)),
                             "popularidad": rng.lognormvariate(0, 0.5)})

    def fila_restaurante(r):
        return [r["id"], r["nombre"] + ("  " if rng.random() < 0.2 else ""), r["categoria"], r["ciudad"],
                r["distrito"].lower() if rng.random() < 0.2 else r["distrito"],
                f"{r['comision']:g}%" if rng.random() < 0.8 else f"{r['comision']:g} %", r["fecha_alta"].isoformat()]

    encabezado_rest = ["id_restaurante", "nombre", "categoria", "ciudad", "distrito", "comision", "fecha_alta"]
    escribir_csv("restaurantes.csv", [encabezado_rest] + [fila_restaurante(r) for r in restaurantes], sep=";")

    # ---------------- clientes ----------------
    NOMBRES = ["María", "José", "Luis", "Carmen", "Juan", "Rosa", "Carlos", "Ana", "Jorge", "Lucía", "Miguel", "Sofía",
               "Diego", "Valeria", "Renzo", "Camila", "Andrés", "Daniela", "Piero", "Fiorella", "Kevin", "Milagros",
               "Bruno", "Ximena", "César", "Gabriela", "Raúl", "Patricia", "Óscar", "Karina"]
    APELLIDOS = ["Quispe", "Flores", "García", "Rodríguez", "Mamani", "Huamán", "Chávez", "Torres", "Rojas", "Vargas",
                 "Ramírez", "Castillo", "Mendoza", "Gutiérrez", "Sánchez", "Díaz", "Cárdenas", "Paredes", "Salazar",
                 "Espinoza"]
    clientes = []
    for i in range(1, 1501):
        nombre, apellido = rng.choice(NOMBRES), rng.choice(APELLIDOS)
        mes_inicio = rng.choices([1, 2, 3, 4, 5, 6], weights=[30, 18, 16, 14, 12, 10])[0] if rng.random() > 0.07 else None
        registro = fecha_al_azar(date(2023, 1, 1), date(2025, mes_inicio, 1) - timedelta(days=1) if mes_inicio else date(2025, 6, 15))
        clientes.append({
            "id": i, "nombre": f"{nombre} {apellido}", "mes_inicio": mes_inicio, "registro": registro,
            "ciudad": rng.choices(["Lima", "Arequipa", "Trujillo"], weights=[65, 20, 15])[0],
            "tasa": rng.lognormvariate(0.75, 0.6), "plan": "prime" if rng.random() < 0.25 else "free",
            "email": f"{sin_tildes(nombre).lower()}.{sin_tildes(apellido).lower()}{i}@"
                     + rng.choice(["gmail.com", "hotmail.com", "outlook.com", "yahoo.es"]),
            "telefono": "9" + "".join(str(rng.randint(0, 9)) for _ in range(8)),
            "actualizado": datetime.combine(registro, datetime.min.time()) + timedelta(minutes=rng.randint(0, 60 * 24 * 10)),
        })

    def fila_cliente(c, plan=None, email=None, actualizado=None):
        tel, r = c["telefono"], rng.random()
        tel = (tel if r < 0.40 else f"+51 {tel[:3]} {tel[3:6]} {tel[6:]}" if r < 0.65 else
               f"{tel[:3]}-{tel[3:6]}-{tel[6:]}" if r < 0.85 else f"(+51) {tel}" if r < 0.95 else
               None if r < 0.97 else "   " if r < 0.985 else tel[:5])
        email = email or c["email"]
        email = None if rng.random() < 0.02 else email.upper() if rng.random() < 0.12 else email
        if email and rng.random() < 0.10:
            email = f" {email}  "
        reg = c["registro"].isoformat() if rng.random() < 0.7 else c["registro"].strftime("%d/%m/%Y")
        return [c["id"], ensuciar(c["nombre"]), email, tel, None if rng.random() < 0.015 else ensuciar(c["ciudad"]),
                ensuciar(plan or c["plan"]), reg, (actualizado or c["actualizado"]).strftime("%Y-%m-%d %H:%M:%S")]

    # 25 clientes que cambiaron de plan: la versión vieja (free) también viene en el archivo
    actualizados = rng.sample([c for c in clientes if c["mes_inicio"]], 25)
    for c in actualizados:
        c["anterior"], c["plan"] = c["actualizado"], "prime"
        c["actualizado"] = c["anterior"] + timedelta(days=rng.randint(20, 120), minutes=rng.randint(0, 600))
    filas = {c["id"]: fila_cliente(c) for c in clientes}
    filas_clientes = list(filas.values())
    filas_clientes += [fila_cliente(c, plan="free", email=c["email"].split("@")[0] + "@yahoo.es", actualizado=c["anterior"])
                       for c in actualizados]
    ids_actualizados = [c["id"] for c in actualizados]
    candidatos = [c["id"] for c in clientes if c["id"] not in ids_actualizados]
    filas_clientes += [list(filas[i]) for i in rng.sample(candidatos, 60)]  # 60 copias exactas
    rng.shuffle(filas_clientes)
    escribir_csv("clientes.csv", [["id_cliente", "nombre", "email", "telefono", "ciudad", "plan", "fecha_registro",
                                   "fecha_actualizacion"]] + filas_clientes)

    # ---------------- pedidos ----------------
    rest_por_ciudad = {ciudad: [r for r in restaurantes if r["ciudad"] == ciudad] for ciudad in DISTRITOS}
    PESOS_HORA = [1, .5, .3, .2, .2, .3, .8, 1.5, 2, 2.5, 3, 5, 9, 10, 7, 4, 3, 4, 6, 9, 10, 8, 5, 2]
    PESOS_PAGO = {"Lima": [38, 32, 15, 15], "Arequipa": [35, 25, 12, 28], "Trujillo": [33, 22, 15, 30]}
    # el ceviche se pide más en verano (enero a marzo); el pollo y la comida criolla, cuando empieza el frío
    TEMPORADA = {"Cevichería": [2.2, 2.2, 1.8, 1.0, 0.7, 0.5], "Pollería": [0.8, 0.8, 0.9, 1.2, 1.5, 1.9],
                 "Criolla": [0.8, 0.8, 0.9, 1.1, 1.3, 1.5]}

    def momento_al_azar(mes):
        dias_mes = ((date(2025, mes + 1, 1) if mes < 12 else date(2026, 1, 1)) - date(2025, mes, 1)).days
        while True:
            dia = date(2025, mes, 1) + timedelta(days=rng.randint(0, dias_mes - 1))
            if dia.weekday() >= 4 or rng.random() < 0.75:  # de viernes a domingo hay más pedidos
                break
        return datetime(dia.year, dia.month, dia.day, rng.choices(range(24), weights=PESOS_HORA)[0],
                        rng.randint(0, 59), rng.randint(0, 59))

    def nuevo_pedido(id_cliente, ciudad, momento):
        opciones = rest_por_ciudad[ciudad] if rng.random() < 0.95 else restaurantes
        pesos = [x["popularidad"] * TEMPORADA.get(x["categoria"], [1] * 6)[momento.month - 1] for x in opciones]
        r = rng.choices(opciones, weights=pesos)[0]
        menu = MENUS[r["categoria"]]
        k = rng.choices([1, 2, 3, 4], weights=[45, 30, 17, 8])[0]
        items = [{"producto": p, "cantidad": rng.choices([1, 2, 3], weights=[70, 22, 8])[0], "precio_unitario": precio}
                 for p, precio in rng.sample(menu, min(k, len(menu)))]
        estado = rng.choices(["entregado", "cancelado", "reembolsado"], weights=[88, 9, 3])[0]
        distancia = round(min(12.0, max(0.3, rng.lognormvariate(1.0, 0.55))), 1)
        minutos = None if estado == "cancelado" else max(8, round(12 + distancia * 4.5 + rng.gauss(0, 6)))
        propina = 0.0 if estado == "cancelado" else float(rng.choices([0, 1, 2, 3, 5], weights=[55, 15, 15, 8, 7])[0])
        if rng.random() < 0.05:
            propina = None
        calificacion = None
        if estado == "entregado" and rng.random() < 0.72:
            base = 5 if minutos < 30 else 4 if minutos < 45 else 3
            calificacion = min(5, max(1, base + rng.choices([-2, -1, 0, 1], weights=[5, 20, 60, 15])[0]))
        return {
            "id_cliente": id_cliente, "id_restaurante": r["id"],
            "id_repartidor": None if estado == "cancelado" else rng.choice(REPARTIDORES[r["ciudad"]]),
            "fecha_hora": momento.strftime("%Y-%m-%dT%H:%M:%S"), "estado": estado,
            "metodo_pago": rng.choices(["yape", "tarjeta", "plin", "efectivo"], weights=PESOS_PAGO[r["ciudad"]])[0],
            "items": items, "entrega": {"distancia_km": distancia, "minutos": minutos, "propina": propina},
            "cupon": rng.choices(["CHASKI10", "BIENVENIDA20", "FINDE15", "YAPEA5", "VERANO2024"],
                                 weights=[35, 20, 20, 15, 10])[0] if rng.random() < 0.14 else None,
            "calificacion": calificacion,
        }

    pedidos = []
    for c in clientes:
        if not c["mes_inicio"]:
            continue
        for mes in range(c["mes_inicio"], 7):
            if mes > c["mes_inicio"] and rng.random() < 0.12:  # el cliente deja de pedir
                break
            n = max(1, poisson(c["tasa"])) if mes == c["mes_inicio"] else poisson(c["tasa"])
            for _ in range(n):
                momento = momento_al_azar(mes)
                pedidos.append((momento, nuevo_pedido(c["id"], c["ciudad"], momento)))
    for _ in range(80):  # pedidos de clientes que no están en clientes.csv
        momento = momento_al_azar(rng.randint(1, 6))
        pedidos.append((momento, nuevo_pedido(9000 + rng.randint(1, 40), rng.choice(list(DISTRITOS)), momento)))
    pedidos.sort(key=lambda x: x[0])

    lineas = {mes: [] for mes in range(1, 7)}
    for n, (momento, p) in enumerate(pedidos, start=1):
        lineas[momento.month].append(json.dumps({"id_pedido": f"CH-{n:06d}", **p}, ensure_ascii=False))
    originales = {mes: list(v) for mes, v in lineas.items()}
    for _ in range(150):  # pedidos reenviados: vuelven a aparecer en el archivo del mes siguiente
        mes = rng.randint(1, 5)
        lineas[mes + 1].append(rng.choice(originales[mes]))
    for k in range(5):  # líneas cortadas a la mitad
        mes = rng.randint(1, 6)
        texto = json.dumps({"id_pedido": f"CH-9{k:05d}", **nuevo_pedido(rng.randint(1, 1500), "Lima", momento_al_azar(mes))},
                           ensure_ascii=False)
        lineas[mes].insert(rng.randint(0, len(lineas[mes])), texto[: int(len(texto) * rng.uniform(0.4, 0.8))])
    for mes, contenido in lineas.items():
        with open(f"{ruta}/pedidos/pedidos_2025_{mes:02d}.json", "w", encoding="utf-8") as f:
            f.write("\n".join(contenido) + "\n")

    # ---------------- actualizaciones de julio ----------------
    cambios = [dict(r, comision=r["comision"] + rng.choice([-3, 2, 3])) for r in rng.sample(restaurantes, 4)]
    cambios[0]["nombre"] += " Express"
    for i, (cat, ciudad) in enumerate([("Sushi", "Lima"), ("Cevichería", "Arequipa")], start=41):
        cambios.append({"id": i, "nombre": rng.choice(NOMBRES_REST[cat]), "categoria": cat, "ciudad": ciudad,
                        "distrito": rng.choice(DISTRITOS[ciudad]), "comision": 15, "fecha_alta": date(2025, 7, 1)})
    escribir_csv("actualizaciones/restaurantes_2025_07.csv", [encabezado_rest] + [fila_restaurante(r) for r in cambios], sep=";")

    print(f"restaurantes.csv: {len(restaurantes)} filas")
    print(f"clientes.csv: {len(filas_clientes)} filas")
    print(f"pedidos/: {len(lineas)} archivos, {sum(len(v) for v in lineas.values())} líneas")
    print(f"actualizaciones/restaurantes_2025_07.csv: {len(cambios)} filas")


generar_datos_chaskifood(RUTA_RAW)

restaurantes.csv: 40 filas
clientes.csv: 1585 filas
pedidos/: 6 archivos, 11683 líneas
actualizaciones/restaurantes_2025_07.csv: 6 filas


In [0]:
display(dbutils.fs.ls(RUTA_RAW))

path,name,size,modificationTime
dbfs:/Volumes/workspace/chaskifood/archivos/raw/actualizaciones/,actualizaciones/,0,1794282440486
dbfs:/Volumes/workspace/chaskifood/archivos/raw/clientes.csv,clientes.csv,164382,1794282438000
dbfs:/Volumes/workspace/chaskifood/archivos/raw/pedidos/,pedidos/,0,1794282440486
dbfs:/Volumes/workspace/chaskifood/archivos/raw/restaurantes.csv,restaurantes.csv,2490,1794282438000


Antes de leer nada con Spark, conviene mirar los archivos tal cual. Los volúmenes se pueden leer también con Python normal (`open`), como cualquier carpeta:

In [0]:
for archivo in ["restaurantes.csv", "clientes.csv", "pedidos/pedidos_2025_01.json"]:
    print(f"==> {archivo}")
    with open(f"{RUTA_RAW}/{archivo}", encoding="utf-8") as f:
        for _ in range(3):
            print(f.readline().rstrip())
    print()

==> restaurantes.csv
id_restaurante;nombre;categoria;ciudad;distrito;comision;fecha_alta
1;El Gallo Fino;Pollería;Lima;San Isidro;20%;2020-12-10
2;Chifa Bambú  ;Chifa;Lima;Barranco;12%;2023-04-17

==> clientes.csv
id_cliente,nombre,email,telefono,ciudad,plan,fecha_registro,fecha_actualizacion
377,Piero Mendoza,piero.mendoza377@hotmail.com,981-253-084,lima,  free ,2023-10-13,2023-10-20 19:15:00
769,andrés huamán,andres.huaman769@gmail.com,941087755,Lima,free,2024-07-17,2024-07-18 04:05:00

==> pedidos/pedidos_2025_01.json
{"id_pedido": "CH-000001", "id_cliente": 1082, "id_restaurante": 8, "id_repartidor": "R01", "fecha_hora": "2025-01-01T00:46:00", "estado": "entregado", "metodo_pago": "yape", "items": [{"producto": "Bowl de quinua", "cantidad": 1, "precio_unitario": 27.9}], "entrega": {"distancia_km": 1.7, "minutos": 18, "propina": 0.0}, "cupon": null, "calificacion": 5}
{"id_pedido": "CH-000002", "id_cliente": 1398, "id_restaurante": 7, "id_repartidor": "R04", "fecha_hora": "2025-01-0

Fíjate en lo que ya se ve: el CSV de restaurantes usa `;` como separador, en clientes hay mayúsculas, espacios de más y teléfonos con formatos distintos, y cada pedido trae una lista `items` y un objeto `entrega` anidados.

## Módulo 1. Repaso: DataFrames a mano

Hay dos tablas chicas que no vienen en archivos: los repartidores y los cupones vigentes. Las creamos con `spark.createDataFrame`. Recuerda que el esquema se puede declarar de dos formas:

```python
# con StructType
StructType([StructField("codigo", StringType(), True), StructField("valor", DoubleType(), True)])

# con un string DDL (más corto)
"codigo STRING, valor DOUBLE"
```

### Ejercicio 1.1: repartidores con `StructType`
Crea el DataFrame `repartidores` con los datos de abajo y un esquema explícito hecho con `StructType`:

| campo | tipo |
|---|---|
| `id_repartidor` | `StringType()` |
| `nombre` | `StringType()` |
| `vehiculo` | `StringType()` |
| `ciudad` | `StringType()` |
| `fecha_ingreso` | `DateType()` |

In [0]:
datos_repartidores = [
    ("R01", "Wilmer Huanca",   "moto",      "Lima",     date(2023, 2, 14)),
    ("R02", "Katherine Soto",  "moto",      "Lima",     date(2023, 6, 1)),
    ("R03", "Jhon Ccori",      "bicicleta", "Lima",     date(2024, 1, 20)),
    ("R04", "Lizbeth Ramos",   "moto",      "Lima",     date(2022, 11, 3)),
    ("R05", "Erick Palomino",  "auto",      "Lima",     date(2024, 8, 12)),
    ("R06", "Yesenia Calle",   "bicicleta", "Lima",     date(2025, 1, 6)),
    ("R07", "Franco Apaza",    "moto",      "Arequipa", date(2023, 4, 9)),
    ("R08", "Nayeli Condori",  "moto",      "Arequipa", date(2024, 3, 15)),
    ("R09", "Alonso Zegarra",  "auto",      "Arequipa", date(2022, 9, 27)),
    ("R10", "Brenda Alvarado", "moto",      "Trujillo", date(2023, 7, 30)),
    ("R11", "Gianfranco Polo", "bicicleta", "Trujillo", date(2024, 5, 18)),
    ("R12", "Rocío Valverde",  "moto",      "Trujillo", date(2023, 12, 2)),
]


esquema_repartidores=StructType([
    StructField("id_repartidor",StringType(),True),
    StructField("nombre",StringType(),True),
    StructField("vehiculo",StringType(),True),
    StructField("ciudad",StringType(),True),
    StructField("fecha_ingreso",DateType(),True)
])
repartidores= spark.createDataFrame(datos_repartidores,schema=esquema_repartidores)


repartidores.printSchema()

root
 |-- id_repartidor: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- vehiculo: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- fecha_ingreso: date (nullable = true)



In [0]:
# comprobación
assert repartidores.count() == 12, "Deberías tener 12 repartidores"
assert repartidores.dtypes == [("id_repartidor", "string"), ("nombre", "string"), ("vehiculo", "string"),
                               ("ciudad", "string"), ("fecha_ingreso", "date")], f"Revisa nombres y tipos: {repartidores.dtypes}"
ok()

Correcto


### Ejercicio 1.2: cupones con esquema DDL
Crea `cupones` con los datos de abajo, esta vez usando un esquema DDL (un string): `codigo STRING, tipo STRING, valor DOUBLE, minimo_compra DOUBLE`.

Cómo funcionan los cupones (lo vamos a necesitar en el módulo 4):
- `tipo = "porcentaje"`: el descuento es `valor` % del subtotal
- `tipo = "monto"`: el descuento es `valor` soles fijos
- en ambos casos, solo aplica si el subtotal del pedido es mayor o igual a `minimo_compra`

In [0]:
datos_cupones = [
    ("CHASKI10",     "porcentaje", 10.0, 0.0),
    ("BIENVENIDA20", "porcentaje", 20.0, 30.0),
    ("FINDE15",      "porcentaje", 15.0, 40.0),
    ("YAPEA5",       "monto",       5.0, 25.0),
]
esquema_ddl = "codigo STRING, tipo STRING, valor DOUBLE, minimo_compra DOUBLE"
cupones = spark.createDataFrame(datos_cupones, esquema_ddl)


display(cupones)

codigo,tipo,valor,minimo_compra
CHASKI10,porcentaje,10.0,0.0
BIENVENIDA20,porcentaje,20.0,30.0
FINDE15,porcentaje,15.0,40.0
YAPEA5,monto,5.0,25.0


In [0]:
# comprobación
assert cupones.count() == 4
assert cupones.dtypes == [("codigo", "string"), ("tipo", "string"), ("valor", "double"), ("minimo_compra", "double")], \
    f"Revisa nombres y tipos: {cupones.dtypes}"
ok()

Correcto


## Módulo 2. Ingesta: de archivos crudos a bronze

Para leer archivos usamos `spark.read` con opciones:

| Opción | Para qué sirve |
|---|---|
| `.option("header", True)` | la primera fila trae los nombres de las columnas |
| `.option("sep", ";")` | el separador no es una coma |
| `.schema("a INT, b STRING")` | esquema explícito |
| `.option("mode", "PERMISSIVE")` | (es el modo por defecto) si una línea no se puede leer, la fila queda con nulos en lugar de romper la lectura |
| `.option("columnNameOfCorruptRecord", "_corrupt_record")` | guarda el texto de la línea que no se pudo leer en esa columna (tiene que estar en el esquema) |

¿Por qué esquema explícito en lugar de `inferSchema`? Porque con `inferSchema` Spark lee los datos una vez más solo para adivinar los tipos, y a veces adivina mal: un teléfono termina como número, un código pierde los ceros a la izquierda, etc. En un pipeline real el esquema se declara.

Todo DataFrame leído de archivos tiene además una columna oculta, `_metadata`, con datos del archivo de origen (`_metadata.file_name`, `_metadata.file_path`, `_metadata.file_modification_time`...). No aparece en `select("*")`, hay que pedirla por nombre. En Unity Catalog reemplaza a la antigua función `input_file_name()`.

Sobre la escritura: Delta valida el esquema de la tabla (schema enforcement). Si una tabla ya existe con otras columnas, un `overwrite` falla con un error de "schema mismatch". Mientras practicamos vamos a agregar `.option("overwriteSchema", True)` para poder re-ejecutar las celdas sin problemas.

### Ejercicio 2.1: restaurantes (CSV con `;`)
Lee `RUTA_RAW + "/restaurantes.csv"` en `restaurantes_raw`, con encabezado, separador `;` y el esquema `ESQUEMA_RESTAURANTES`. La comisión viene como texto (por ejemplo `"18%"`), la limpiamos en el módulo 3.

In [0]:
ESQUEMA_RESTAURANTES = "id_restaurante INT, nombre STRING, categoria STRING, ciudad STRING, distrito STRING, comision STRING, fecha_alta DATE"
restaurantes_raw=spark.read.option('header','true').option('sep',';').schema(ESQUEMA_RESTAURANTES).csv("/Volumes/workspace/chaskifood/archivos/raw/restaurantes.csv")

display(restaurantes_raw)

id_restaurante,nombre,categoria,ciudad,distrito,comision,fecha_alta
1,El Gallo Fino,Pollería,Lima,San Isidro,20%,2020-12-10
2,Chifa Bambú,Chifa,Lima,Barranco,12%,2023-04-17
3,Mar Picante,Cevichería,Lima,Los Olivos,12%,2023-08-13
4,Pizza Nostra,Pizzería,Lima,barranco,12%,2021-07-22
5,Doble Queso,Hamburguesas,Lima,La Molina,12 %,2020-03-29
6,Nikkei Roll,Sushi,Lima,miraflores,12%,2020-07-11
7,La Sazón de Mamá Rosa,Criolla,Lima,san miguel,18%,2023-12-19
8,Raíz Bowl,Vegetariana,Lima,La Molina,20%,2024-02-13
9,El Gallo Fino,Pollería,Lima,Los Olivos,22.5%,2019-04-27
10,Chifa Loto Azul,Chifa,Lima,La Molina,22.5%,2022-07-02


In [0]:
# comprobación
assert restaurantes_raw.columns == ["id_restaurante", "nombre", "categoria", "ciudad", "distrito", "comision", "fecha_alta"]
assert restaurantes_raw.count() == 40, "Deberían ser 40 restaurantes"
assert restaurantes_raw.filter(F.col("categoria").isNull()).count() == 0, "Hay categorías nulas: ¿pusiste sep=';'?"
ok()

Correcto


### Ejercicio 2.2: clientes (CSV)
Lee `RUTA_RAW + "/clientes.csv"` en `clientes_raw`, con encabezado y el esquema `ESQUEMA_CLIENTES`. Casi todo se lee como texto a propósito: viene sucio y lo vamos a convertir después, ya limpio.

In [0]:
ESQUEMA_CLIENTES = ("id_cliente INT, nombre STRING, email STRING, telefono STRING, ciudad STRING, "
                    "plan STRING, fecha_registro STRING, fecha_actualizacion STRING")
clientes_raw=spark.read.option('header','true').schema(ESQUEMA_CLIENTES).csv("/Volumes/workspace/chaskifood/archivos/raw/clientes.csv")

display(clientes_raw.limit(10))

id_cliente,nombre,email,telefono,ciudad,plan,fecha_registro,fecha_actualizacion
377,Piero Mendoza,piero.mendoza377@hotmail.com,981-253-084,lima,free,2023-10-13,2023-10-20 19:15:00
769,andrés huamán,andres.huaman769@gmail.com,941087755,Lima,free,2024-07-17,2024-07-18 04:05:00
88,Rosa Mamani,rosa.mamani88@yahoo.es,943-425-682,Lima,free,02/06/2024,2024-06-07 04:29:00
539,Sofía Huamán,sofia.huaman539@gmail.com,+51 999 397 538,Lima,FREE,2024-02-03,2024-02-11 22:34:00
1126,Milagros Quispe,MILAGROS.QUISPE1126@YAHOO.ES,(+51) 939223407,Trujillo,free,2023-05-27,2023-05-27 16:02:00
200,Juan Vargas,juan.vargas200@yahoo.es,968973906,Lima,PRIME,2024-03-30,2024-03-30 04:07:00
101,José Quispe,jose.quispe101@hotmail.com,947220777,Lima,free,2024-04-10,2024-04-18 13:45:00
125,Gabriela Espinoza,gabriela.espinoza125@outlook.com,997757692,LIMA,FREE,2023-08-17,2023-08-25 13:26:00
666,Diego Torres,diego.torres666@yahoo.es,930218761,Lima,FREE,28/03/2023,2023-03-31 00:03:00
759,Rosa Díaz,rosa.diaz759@outlook.com,+51 946 039 884,Lima,free,2023-04-12,2023-04-17 01:18:00


In [0]:
# comprobación
assert clientes_raw.columns == ["id_cliente", "nombre", "email", "telefono", "ciudad", "plan", "fecha_registro", "fecha_actualizacion"]
assert clientes_raw.count() == 1585, f"Esperaba 1585 filas, hay {clientes_raw.count()}"
ok()

Correcto


### Ejercicio 2.3: pedidos (JSON anidado en varios archivos)
Los pedidos vienen en 6 archivos JSON Lines (un objeto JSON por línea) dentro de la carpeta `RUTA_RAW + "/pedidos"`. Si le pasas a Spark la carpeta, lee todos los archivos que tenga.

Cada pedido tiene campos anidados: `items` es una lista de productos (`ARRAY<STRUCT<...>>`) y `entrega` es un objeto (`STRUCT<...>`). El esquema ya está escrito en `ESQUEMA_PEDIDOS`, fíjate cómo se declaran esos tipos. Incluye la columna `_corrupt_record` para las líneas que no se puedan leer.

Crea `pedidos_raw`:
1. leyendo la carpeta con ese esquema y la opción `columnNameOfCorruptRecord`
2. agregando la columna `archivo` con `_metadata.file_name`
3. agregando la columna `fecha_ingesta` con `F.current_timestamp()`

Pista: `.select("*", F.col("_metadata.file_name").alias("archivo"))`

In [0]:
ESQUEMA_PEDIDOS = """
    id_pedido STRING,
    id_cliente INT,
    id_restaurante INT,
    id_repartidor STRING,
    fecha_hora STRING,
    estado STRING,
    metodo_pago STRING,
    items ARRAY<STRUCT<producto: STRING, cantidad: INT, precio_unitario: DOUBLE>>,
    entrega STRUCT<distancia_km: DOUBLE, minutos: INT, propina: DOUBLE>,
    cupon STRING,
    calificacion INT,
    _corrupt_record STRING
"""
pedidos_raw = spark.read.option("columnNameOfCorruptRecord", "_corrupt_record") \
                        .schema(ESQUEMA_PEDIDOS) \
                        .json("/Volumes/workspace/chaskifood/archivos/raw/pedidos/") \
                        .select("*", F.col("_metadata.file_name").alias("archivo")) \
                        .withColumn("fecha_ingesta", F.current_timestamp())

display(pedidos_raw.limit(5))

id_pedido,id_cliente,id_restaurante,id_repartidor,fecha_hora,estado,metodo_pago,items,entrega,cupon,calificacion,_corrupt_record,archivo,fecha_ingesta
CH-006644,1073,4,R02,2025-05-01T00:39:19,entregado,tarjeta,"List(List(Pan al ajo, 1, 11.9))","List(1.4, 14, 0.0)",null,null,null,pedidos_2025_05.json,2026-11-10T03:47:36.633Z
CH-006645,162,19,R02,2025-05-01T01:24:41,entregado,tarjeta,"List(List(Chicharrón de pescado, 1, 33.9), List(Leche de tigre, 2, 18.9))","List(2.3, 15, 2.0)",FINDE15,4,null,pedidos_2025_05.json,2026-11-10T03:47:36.633Z
CH-006646,939,34,R10,2025-05-01T08:01:42,entregado,efectivo,"List(List(Chancho al tamarindo, 1, 32.9), List(Tallarín saltado, 2, 26.9))","List(6.8, 42, 1.0)",null,null,null,pedidos_2025_05.json,2026-11-10T03:47:36.633Z
CH-006647,1438,7,R03,2025-05-01T08:39:19,entregado,efectivo,"List(List(Chicha morada 1L, 2, 12.0), List(Ají de gallina, 1, 28.9), List(Lomo saltado, 3, 36.9))","List(4.1, 34, 2.0)",CHASKI10,4,null,pedidos_2025_05.json,2026-11-10T03:47:36.633Z
CH-006648,83,24,R05,2025-05-01T09:19:00,entregado,efectivo,"List(List(Bowl de quinua, 1, 27.9))","List(1.4, 12, 5.0)",null,5,null,pedidos_2025_05.json,2026-11-10T03:47:36.633Z


In [0]:
# comprobación
assert pedidos_raw.count() == 11683, "Deberías leer todas las líneas de los 6 archivos (11683)"
assert {"archivo", "fecha_ingesta", "_corrupt_record"} <= set(pedidos_raw.columns)
assert pedidos_raw.select("archivo").distinct().count() == 6, "La columna archivo debería tener 6 valores distintos"
assert dict(pedidos_raw.dtypes)["items"].startswith("array<struct"), "Usa ESQUEMA_PEDIDOS"
ok()

Correcto


### Ejercicio 2.4: guardar la capa bronze
Guarda los tres DataFrames como tablas Delta (modo `overwrite` y con `overwriteSchema`). Recorre el diccionario `tablas_bronze` con un `for` en vez de escribir tres veces lo mismo. Después lee la tabla `bronze_pedidos` en la variable `bronze_pedidos`.

In [0]:
tablas_bronze = {
    "bronze_restaurantes": restaurantes_raw,
    "bronze_clientes": clientes_raw,
    "bronze_pedidos": pedidos_raw,
}
for nombre_tabla, df in tablas_bronze.items():
    df.write \
      .format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(nombre_tabla)
bronze_pedidos = spark.table("bronze_pedidos")

display(bronze_pedidos.limit(5))

id_pedido,id_cliente,id_restaurante,id_repartidor,fecha_hora,estado,metodo_pago,items,entrega,cupon,calificacion,_corrupt_record,archivo,fecha_ingesta
CH-000001,1082,8,R01,2025-01-01T00:46:00,entregado,yape,"List(List(Bowl de quinua, 1, 27.9))","List(1.7, 18, 0.0)",null,5,null,pedidos_2025_01.json,2026-11-10T03:47:48.060Z
CH-000002,1398,7,R04,2025-01-01T04:20:22,entregado,yape,"List(List(Causa limeña, 2, 19.9), List(Arroz con pollo, 1, 26.9))","List(1.2, 30, 2.0)",null,4,null,pedidos_2025_01.json,2026-11-10T03:47:48.060Z
CH-000003,910,22,R02,2025-01-01T06:38:42,entregado,yape,"List(List(Maki furai, 2, 29.9), List(Nigiri mixto, 1, 36.9))","List(2.9, 31, 0.0)",null,4,null,pedidos_2025_01.json,2026-11-10T03:47:48.060Z
CH-000004,120,10,R05,2025-01-01T09:21:03,entregado,tarjeta,"List(List(Wantán frito, 1, 14.9), List(Tallarín saltado, 1, 26.9))","List(2.3, 31, 0.0)",null,4,null,pedidos_2025_01.json,2026-11-10T03:47:48.060Z
CH-000005,891,4,R04,2025-01-01T09:58:33,entregado,efectivo,"List(List(Pizza americana familiar, 1, 45.9))","List(2.9, 20, 1.0)",null,5,null,pedidos_2025_01.json,2026-11-10T03:47:48.060Z


In [0]:
# comprobación
for nombre in tablas_bronze:
    assert spark.catalog.tableExists(f"{CATALOGO}.{ESQUEMA}.{nombre}"), f"No existe la tabla {nombre}"
assert bronze_pedidos.count() == 11683
ok("Correcto. Las tablas aparecen en Catalog > workspace > chaskifood")

Correcto. Las tablas aparecen en Catalog > workspace > chaskifood


### Ejercicio 2.5: revisar la ingesta
A partir de `bronze_pedidos`:
- guarda en `n_corruptos` cuántas filas tienen `_corrupt_record` no nulo
- crea `por_archivo` con el número de filas por `archivo` (columna `n_filas`), ordenado por archivo

Dato: si intentas el mismo conteo directamente sobre `pedidos_raw`, Spark puede rechazarlo con el error `QUERY_ONLY_CORRUPT_RECORD_COLUMN`, porque no permite consultar sobre el archivo crudo solo la columna `_corrupt_record`. Guardar primero en bronze evita el problema, y además es lo que se hace en un pipeline real: guardar lo que llegó, tal cual, antes de transformarlo.

In [0]:
n_corruptos = bronze_pedidos.filter(F.col("_corrupt_record").isNotNull()).count()

por_archivo=bronze_pedidos.groupBy("archivo").agg(F.count("*").alias("n_filas")).orderBy("archivo")
print(f"Líneas corruptas: {n_corruptos}")
display(por_archivo)
display(bronze_pedidos.filter(F.col("_corrupt_record").isNotNull()).select("archivo", "_corrupt_record"))

Líneas corruptas: 5


archivo,n_filas
pedidos_2025_01.json,1019
pedidos_2025_02.json,1483
pedidos_2025_03.json,1920
pedidos_2025_04.json,2309
pedidos_2025_05.json,2491
pedidos_2025_06.json,2461


archivo,_corrupt_record
pedidos_2025_01.json,"{""id_pedido"": ""CH-900001"", ""id_cliente"": 892, ""id_restaurante"": 4, ""id_repartidor"": ""R03"", ""fecha_hora"": ""2025-01-03T15:48:04"", ""estado"": ""entregado"", ""metodo_pago"": ""efectivo"", ""items"": [{""producto"": ""Pizza hawa"
pedidos_2025_01.json,"{""id_pedido"": ""CH-900003"", ""id_cliente"": 1256, ""id_restaurante"": 19, ""id_repartidor"": ""R03"", ""fecha_hora"": ""2025-01-29T20:02:47"", ""estado"": ""entregado"", ""metodo_pago"": ""yape"", ""items"": [{""producto"": ""Cev"
pedidos_2025_02.json,"{""id_pedido"": ""CH-900002"", ""id_cliente"": 827, ""id_restaurante"": 3, ""id_repartidor"": ""R06"", ""fecha_hora"": ""2025-02-21T11:36:10"", ""estado"": ""entregado"", ""metodo_pago"": ""tarjeta"", ""items"": [{""producto"": ""Chicharrón de pescado"", ""cantidad"": 1, ""precio_unitario"": 33.9}, {""producto"": ""Ceviche clásico"", ""cantidad"": 1, ""precio_unitario"""
pedidos_2025_03.json,"{""id_pedido"": ""CH-900000"", ""id_cliente"": 525, ""id_restaurante"": 4, ""id_repartidor"": ""R01"", ""fecha_hora"": ""2025-03-16T19:10:17"", ""estado"": ""entregado"", ""metodo_p"
pedidos_2025_06.json,"{""id_pedido"": ""CH-900004"", ""id_cliente"": 937, ""id_restaurante"": 13, ""id_repartidor"": ""R01"", ""fecha_hora"": ""2025-06-20T18:53:09"", ""estado"": ""entregado"", ""metodo_pago"": ""tarjeta"", ""items"": [{""producto"": ""Milkshake de lúcuma"""


In [0]:
# comprobación
assert n_corruptos == 5, f"Esperaba 5 líneas corruptas, hay {n_corruptos}"
assert por_archivo.columns == ["archivo", "n_filas"] and por_archivo.count() == 6
assert por_archivo.agg(F.sum("n_filas")).first()[0] == bronze_pedidos.count()
archivos = [r["archivo"] for r in por_archivo.collect()]
assert archivos == sorted(archivos), "Ordena por archivo"
ok()

Correcto


## Módulo 3. Silver: limpiar clientes y restaurantes

Ahora pasamos de bronze a silver. En la práctica 1 usamos `trim`, `initcap`, `lower`, `fillna` y `try_cast`. Aquí se suman:

| Problema | Qué usar |
|---|---|
| quitar todo lo que no sea dígito | `F.regexp_replace("telefono", r"[^0-9]", "")` |
| extraer un patrón | `F.regexp_extract(col, r"(9\d{8})$", 1)` devuelve lo que calza con el grupo 1, o `""` si no hay coincidencia |
| convertir `""` en nulo | `F.nullif(col, F.lit(""))` |
| fechas en un formato conocido | `F.try_to_timestamp(col, F.lit("dd/MM/yyyy"))` devuelve `null` si el texto no calza con el formato |
| quedarse con el primer valor no nulo | `F.coalesce(a, b, ...)` |
| la última versión de cada registro | `row_number()` sobre una ventana particionada por la clave y ordenada por fecha descendente |

Con ANSI activo, `F.to_date("17/05/2024")` sin formato lanza un error. Por eso usamos `try_to_timestamp` con un formato explícito.

También vamos a empaquetar la limpieza en funciones. `df.transform(funcion)` aplica una función que recibe un DataFrame y devuelve otro. Así los pasos se pueden encadenar y reutilizar:

In [0]:
def agregar_igv(df, tasa=0.18):
    return df.withColumn("precio_con_igv", F.round(F.col("precio") * (1 + tasa), 2))

productos = spark.createDataFrame([("Lomo saltado", 36.9), ("Ceviche clásico", 35.9)], "producto STRING, precio DOUBLE")
display(productos.transform(agregar_igv))
display(productos.transform(agregar_igv, tasa=0.10))   # los argumentos extra se pasan a la función

producto,precio,precio_con_igv
Lomo saltado,36.9,43.54
Ceviche clásico,35.9,42.36


producto,precio,precio_con_igv
Lomo saltado,36.9,40.59
Ceviche clásico,35.9,39.49


En este módulo trabajamos sobre las tablas bronze que guardaste en el módulo 2:

In [0]:
bronze_clientes = spark.read.table("bronze_clientes")
bronze_restaurantes = spark.read.table("bronze_restaurantes")

### Ejercicio 3.1: perfil de calidad
Antes de limpiar conviene medir. Crea `perfil`, un DataFrame de una fila que tenga, para cada columna de texto de `bronze_clientes`, cuántos valores son nulos o están en blanco (solo espacios). Cada columna del resultado se llama igual que la columna original.

Pistas:
- las columnas de texto salen de `bronze_clientes.dtypes`: `[c for c, t in bronze_clientes.dtypes if t == "string"]`
- `(F.col(c).isNull() | (F.trim(c) == "")).cast("int")` vale 1 en los casos malos; súmalo con `F.sum` en una list comprehension

In [0]:
import pyspark.sql.functions as F
columnas = [c for c, t in bronze_clientes.dtypes if t == "string"]
perfil = bronze_clientes.select(
    [F.sum((F.col(c).isNull() | (F.trim(F.col(c)) == "")).cast("int")).alias(c) for c in columnas]
)
display(perfil)

nombre,email,telefono,ciudad,plan,fecha_registro,fecha_actualizacion
0,40,60,21,0,0,0


In [0]:
# comprobación
fila = perfil.first()
assert "id_cliente" not in perfil.columns, "Solo columnas de texto"
assert len(perfil.columns) == 7
assert (fila["nombre"], fila["email"], fila["telefono"], fila["ciudad"]) == (0, 40, 60, 21), \
    f"Revisa el conteo (nulos o en blanco): {fila.asDict()}"
ok()

Correcto


### Ejercicio 3.2: duplicados
Calcula:
- `n_duplicados_exactos`: filas de `bronze_clientes` menos filas distintas
- `n_ids_repetidos`: cuántos `id_cliente` aparecen más de una vez

¿Por qué no salen iguales? Porque algunos clientes aparecen dos veces con datos distintos (por ejemplo, alguien que pasó de free a prime). Esos no son duplicados exactos y `distinct()` no los elimina.

In [0]:
n_duplicados_exactos = bronze_clientes.count() - bronze_clientes.distinct().count()

n_ids_repetidos= bronze_clientes.groupBy("id_cliente").count().filter(F.col("count") > 1).count()
# n_ids_repetidos = bronze_clientes.groupBy("id_cliente").count().filter(F.col("count") > 1).count()
print(n_duplicados_exactos, n_ids_repetidos)

60 85


In [0]:
# comprobación
assert n_duplicados_exactos == 60, f"Esperaba 60 duplicados exactos, tienes {n_duplicados_exactos}"
assert n_ids_repetidos == 85, f"Esperaba 85 ids repetidos, tienes {n_ids_repetidos}"
ok()

Correcto


### Ejercicio 3.3: normalizar textos
A partir de `bronze_clientes`, crea `clientes_norm`:
- `nombre`: sin espacios de sobra y con mayúscula inicial en cada palabra
- `email`: sin espacios y en minúsculas
- `ciudad`: sin espacios y con mayúscula inicial; los nulos pasan a `"Sin ciudad"`
- `plan`: sin espacios y en minúsculas

In [0]:

clientes_norm = bronze_clientes \
    .withColumn("nombre", F.initcap(F.trim(F.col("nombre")))) \
    .withColumn("email", F.lower(F.trim(F.col("email")))) \
    .withColumn("ciudad", F.coalesce(F.initcap(F.trim(F.col("ciudad"))), F.lit("Sin ciudad"))) \
    .withColumn("plan", F.lower(F.trim(F.col("plan"))))

display(clientes_norm.limit(10))

id_cliente,nombre,email,telefono,ciudad,plan,fecha_registro,fecha_actualizacion
377,Piero Mendoza,piero.mendoza377@hotmail.com,981-253-084,Lima,free,2023-10-13,2023-10-20 19:15:00
769,Andrés Huamán,andres.huaman769@gmail.com,941087755,Lima,free,2024-07-17,2024-07-18 04:05:00
88,Rosa Mamani,rosa.mamani88@yahoo.es,943-425-682,Lima,free,02/06/2024,2024-06-07 04:29:00
539,Sofía Huamán,sofia.huaman539@gmail.com,+51 999 397 538,Lima,free,2024-02-03,2024-02-11 22:34:00
1126,Milagros Quispe,milagros.quispe1126@yahoo.es,(+51) 939223407,Trujillo,free,2023-05-27,2023-05-27 16:02:00
200,Juan Vargas,juan.vargas200@yahoo.es,968973906,Lima,prime,2024-03-30,2024-03-30 04:07:00
101,José Quispe,jose.quispe101@hotmail.com,947220777,Lima,free,2024-04-10,2024-04-18 13:45:00
125,Gabriela Espinoza,gabriela.espinoza125@outlook.com,997757692,Lima,free,2023-08-17,2023-08-25 13:26:00
666,Diego Torres,diego.torres666@yahoo.es,930218761,Lima,free,28/03/2023,2023-03-31 00:03:00
759,Rosa Díaz,rosa.diaz759@outlook.com,+51 946 039 884,Lima,free,2023-04-12,2023-04-17 01:18:00


In [0]:
# comprobación
ciudades = {r["ciudad"] for r in clientes_norm.select("ciudad").distinct().collect()}
planes = {r["plan"] for r in clientes_norm.select("plan").distinct().collect()}
assert ciudades == {"Lima", "Arequipa", "Trujillo", "Sin ciudad"}, f"Ciudades: {ciudades}"
assert planes == {"free", "prime"}, f"Planes: {planes}"
assert clientes_norm.filter(F.col("email") != F.lower(F.trim("email"))).count() == 0, "Revisa el email"
assert clientes_norm.filter(F.col("nombre") != F.initcap(F.trim("nombre"))).count() == 0, "Revisa el nombre"
assert clientes_norm.count() == bronze_clientes.count(), "Todavía no quites filas"
ok()

Correcto


### Ejercicio 3.4: teléfonos con expresiones regulares
A partir de `clientes_norm`, crea `clientes_tel` reemplazando `telefono` por un número de 9 dígitos que empiece con 9:
1. quita todo lo que no sea dígito
2. quédate con los últimos 9 dígitos, si empiezan con 9: `F.regexp_extract(..., r"(9\d{8})$", 1)`
3. si no hubo coincidencia (`regexp_extract` devuelve `""`), el teléfono queda en `null`

| Viene así | Queda así |
|---|---|
| `"+51 987 654 321"` | `"987654321"` |
| `"987-654-321"` | `"987654321"` |
| `"(+51) 987654321"` | `"987654321"` |
| `"98765"` o `"   "` | `null` |

In [0]:

clientes_tel = clientes_norm.withColumn(
    "telefono",
    F.expr("""
        nullif(
            regexp_extract(
                regexp_replace(telefono, '[^0-9]', ''), 
                r"(9\d{8})$", 
                1
            ), 
            ''
        )
    """)
)

display(clientes_tel.limit(10))

<>:4: SyntaxWarning: invalid escape sequence '\d'
<>:4: SyntaxWarning: invalid escape sequence '\d'
/home/spark-cea34f35-1d54-497e-a9db-ec/.ipykernel/63/command-5989627092665136-1527748226:4: SyntaxWarning: invalid escape sequence '\d'
  F.expr("""
<unknown>:4: SyntaxWarning: invalid escape sequence '\d'


id_cliente,nombre,email,telefono,ciudad,plan,fecha_registro,fecha_actualizacion
377,Piero Mendoza,piero.mendoza377@hotmail.com,981253084,Lima,free,2023-10-13,2023-10-20 19:15:00
769,Andrés Huamán,andres.huaman769@gmail.com,941087755,Lima,free,2024-07-17,2024-07-18 04:05:00
88,Rosa Mamani,rosa.mamani88@yahoo.es,943425682,Lima,free,02/06/2024,2024-06-07 04:29:00
539,Sofía Huamán,sofia.huaman539@gmail.com,999397538,Lima,free,2024-02-03,2024-02-11 22:34:00
1126,Milagros Quispe,milagros.quispe1126@yahoo.es,939223407,Trujillo,free,2023-05-27,2023-05-27 16:02:00
200,Juan Vargas,juan.vargas200@yahoo.es,968973906,Lima,prime,2024-03-30,2024-03-30 04:07:00
101,José Quispe,jose.quispe101@hotmail.com,947220777,Lima,free,2024-04-10,2024-04-18 13:45:00
125,Gabriela Espinoza,gabriela.espinoza125@outlook.com,997757692,Lima,free,2023-08-17,2023-08-25 13:26:00
666,Diego Torres,diego.torres666@yahoo.es,930218761,Lima,free,28/03/2023,2023-03-31 00:03:00
759,Rosa Díaz,rosa.diaz759@outlook.com,946039884,Lima,free,2023-04-12,2023-04-17 01:18:00


In [0]:
# comprobación
mal = clientes_tel.filter(F.col("telefono").isNotNull() & ~F.col("telefono").rlike(r"^9\d{8}$")).count()
assert mal == 0, f"{mal} teléfonos no tienen 9 dígitos empezando en 9"
nulos = clientes_tel.filter(F.col("telefono").isNull()).count()
assert nulos == 88, f"Esperaba 88 teléfonos nulos (vacíos o incompletos), tienes {nulos}"
ok()

Correcto


### Ejercicio 3.5: fechas en dos formatos
A partir de `clientes_tel`, crea `clientes_fechas`:
- `fecha_registro` como `DATE`. Viene como `"2024-05-17"` o como `"17/05/2024"`: intenta cada formato con `F.try_to_timestamp` y quédate con el que funcione usando `F.coalesce`. Luego pásalo a fecha con `F.to_date`.
- `fecha_actualizacion` como `TIMESTAMP`. Viene como `"2025-03-01 14:20:00"`, que `F.to_timestamp` entiende sin indicar formato.

In [0]:

clientes_fechas = clientes_tel.withColumn(
    "fecha_registro",
    F.to_date(
        F.coalesce(
            F.try_to_timestamp(F.col("fecha_registro"), F.lit("yyyy-MM-dd")),
            F.try_to_timestamp(F.col("fecha_registro"), F.lit("dd/MM/yyyy"))
        )
    )
).withColumn(
    "fecha_actualizacion",
    F.to_timestamp(F.col("fecha_actualizacion"))
)

display(clientes_fechas.select("id_cliente", "fecha_registro", "fecha_actualizacion").limit(10))

id_cliente,fecha_registro,fecha_actualizacion
377,2023-10-13,2023-10-20T19:15:00.000Z
769,2024-07-17,2024-07-18T04:05:00.000Z
88,2024-06-02,2024-06-07T04:29:00.000Z
539,2024-02-03,2024-02-11T22:34:00.000Z
1126,2023-05-27,2023-05-27T16:02:00.000Z
200,2024-03-30,2024-03-30T04:07:00.000Z
101,2024-04-10,2024-04-18T13:45:00.000Z
125,2023-08-17,2023-08-25T13:26:00.000Z
666,2023-03-28,2023-03-31T00:03:00.000Z
759,2023-04-12,2023-04-17T01:18:00.000Z


In [0]:
# comprobación
tipos = dict(clientes_fechas.dtypes)
assert tipos["fecha_registro"] == "date" and tipos["fecha_actualizacion"] == "timestamp", f"Tipos: {tipos}"
assert clientes_fechas.filter(F.col("fecha_registro").isNull()).count() == 0, "Hay fechas que no se pudieron convertir"
r = clientes_fechas.agg(F.min("fecha_registro"), F.max("fecha_registro")).first()
assert r[0] >= date(2023, 1, 1) and r[1] <= date(2025, 6, 30), f"Rango raro: {r[0]} a {r[1]}. ¿Confundiste día y mes?"
ok()

Correcto


### Ejercicio 3.6: quedarse con la versión más reciente
Crea `silver_clientes_df` con una sola fila por `id_cliente`: la de `fecha_actualizacion` más reciente. Usa una ventana con `row_number` y no dejes la columna auxiliar al final.

Después guárdalo como la tabla `silver_clientes` y léela de nuevo en la variable `silver_clientes`.

In [0]:
from pyspark.sql.window import Window

ventana_clientes = Window.partitionBy("id_cliente").orderBy(F.col("fecha_actualizacion").desc())

silver_clientes_df = clientes_fechas.withColumn("rn", F.row_number().over(ventana_clientes))\
    .filter(F.col("rn") == 1).drop("rn")

silver_clientes_df.write \
                  .format("delta") \
                  .mode("overwrite") \
                  .option("overwriteSchema", "true") \
                  .saveAsTable("silver_clientes")
silver_clientes = spark.table("silver_clientes")
display(silver_clientes.limit(10))

id_cliente,nombre,email,telefono,ciudad,plan,fecha_registro,fecha_actualizacion
1,Óscar Mamani,oscar.mamani1@outlook.com,942540989,Lima,free,2023-04-13,2023-04-14T00:51:00.000Z
2,Sofía Vargas,sofia.vargas2@hotmail.com,940440346,Arequipa,prime,2024-07-28,2024-07-29T13:55:00.000Z
3,José Vargas,jose.vargas3@yahoo.es,988275612,Arequipa,free,2024-10-08,2024-10-12T20:02:00.000Z
4,Milagros Ramírez,milagros.ramirez4@outlook.com,973508616,Arequipa,free,2023-04-27,2023-05-03T15:39:00.000Z
5,Juan Flores,juan.flores5@hotmail.com,908438007,Arequipa,free,2024-01-21,2024-01-30T13:46:00.000Z
6,Ximena Salazar,ximena.salazar6@yahoo.es,905726326,Lima,free,2025-05-25,2025-05-27T22:26:00.000Z
7,Karina Chávez,karina.chavez7@yahoo.es,949838836,Lima,free,2023-04-04,2023-04-07T00:08:00.000Z
8,Diego Chávez,diego.chavez8@yahoo.es,929660208,Arequipa,free,2024-07-03,2024-07-09T04:49:00.000Z
9,Carmen Flores,carmen.flores9@yahoo.es,949481846,Lima,free,2024-08-22,2024-08-28T21:44:00.000Z
10,Milagros Ramírez,milagros.ramirez10@outlook.com,979151752,Lima,free,2024-03-18,2024-03-26T16:39:00.000Z


In [0]:
# comprobación
assert silver_clientes.count() == 1500, f"Deberían quedar 1500 clientes, hay {silver_clientes.count()}"
assert silver_clientes.select("id_cliente").distinct().count() == 1500
assert silver_clientes.columns == bronze_clientes.columns, "No dejes columnas auxiliares"
ultima = clientes_fechas.groupBy("id_cliente").agg(F.max("fecha_actualizacion").alias("ultima"))
assert silver_clientes.join(ultima, "id_cliente").filter(F.col("fecha_actualizacion") != F.col("ultima")).count() == 0, \
    "Para algún cliente no te quedaste con la versión más reciente"
ok()

Correcto


### Ejercicio 3.7: restaurantes con una función reutilizable
Escribe la función `limpiar_restaurantes(df)`. Recibe un DataFrame con las columnas de `bronze_restaurantes` y devuelve uno con:
- `nombre`: sin espacios de sobra
- `distrito`: sin espacios y con mayúscula inicial
- `comision_pct`: la comisión como número (`"18%"` pasa a `18.0` y `"22.5 %"` a `22.5`). Pista: borra todo lo que no sea dígito o punto y castea a `double`
- las columnas en este orden: `id_restaurante`, `nombre`, `categoria`, `ciudad`, `distrito`, `comision_pct`, `fecha_alta` (la columna `comision` original ya no va)

Aplícala con `bronze_restaurantes.transform(limpiar_restaurantes)` y guarda el resultado como la tabla `silver_restaurantes`. La vamos a reutilizar en el módulo 9.

In [0]:
def limpiar_restaurantes(df):
    df = df.withColumn("nombre", F.trim(F.col("nombre"))) \
           .withColumn("distrito", F.initcap(F.trim(F.col("distrito")))) \
           .withColumn(
               "comision_pct",
               F.regexp_replace(F.col("comision"), "[^0-9.]", "").cast("double")
           ) \
           .select(
               "id_restaurante",
               "nombre",
               "categoria",
               "ciudad",
               "distrito",
               "comision_pct",
               "fecha_alta"
           )
    return df

silver_restaurantes_df = bronze_restaurantes.transform(limpiar_restaurantes)
silver_restaurantes_df.write \
                      .format("delta") \
                      .mode("overwrite") \
                      .option("overwriteSchema", "true") \
                      .saveAsTable("silver_restaurantes")

silver_restaurantes = spark.read.table("silver_restaurantes")
display(silver_restaurantes.limit(10))

id_restaurante,nombre,categoria,ciudad,distrito,comision_pct,fecha_alta
1,El Gallo Fino,Pollería,Lima,San Isidro,20.0,2020-12-10
2,Chifa Bambú,Chifa,Lima,Barranco,12.0,2023-04-17
3,Mar Picante,Cevichería,Lima,Los Olivos,12.0,2023-08-13
4,Pizza Nostra,Pizzería,Lima,Barranco,12.0,2021-07-22
5,Doble Queso,Hamburguesas,Lima,La Molina,12.0,2020-03-29
6,Nikkei Roll,Sushi,Lima,Miraflores,12.0,2020-07-11
7,La Sazón de Mamá Rosa,Criolla,Lima,San Miguel,18.0,2023-12-19
8,Raíz Bowl,Vegetariana,Lima,La Molina,20.0,2024-02-13
9,El Gallo Fino,Pollería,Lima,Los Olivos,22.5,2019-04-27
10,Chifa Loto Azul,Chifa,Lima,La Molina,22.5,2022-07-02


In [0]:
# comprobación
columnas = ["id_restaurante", "nombre", "categoria", "ciudad", "distrito", "comision_pct", "fecha_alta"]
assert limpiar_restaurantes(bronze_restaurantes).columns == columnas, "Revisa las columnas que devuelve la función"
assert silver_restaurantes.columns == columnas and silver_restaurantes.count() == 40
assert dict(silver_restaurantes.dtypes)["comision_pct"] == "double"
assert silver_restaurantes.filter(~F.col("comision_pct").between(5, 30)).count() == 0, \
    "comision_pct debe ser un número como 18.0 (no 0.18 ni '18%')"
assert silver_restaurantes.filter((F.col("nombre") != F.trim("nombre")) | (F.col("distrito") != F.initcap("distrito"))).count() == 0
ok()

Correcto


## Módulo 4. Silver: pedidos con datos anidados

Los pedidos traen dos tipos complejos:
- `entrega` es un `STRUCT`: un objeto con campos adentro
- `items` es un `ARRAY<STRUCT>`: una lista de objetos

| Para | Se usa |
|---|---|
| leer un campo de un struct | `F.col("entrega.minutos")` o `F.col("entrega")["minutos"]` |
| traer todos los campos de un struct | `df.select("entrega.*")` |
| cantidad de elementos de un array | `F.size("items")` |
| una fila por cada elemento del array | `F.explode("items")` (`F.explode_outer` si quieres conservar las filas con arrays vacíos o nulos) |
| operar dentro del array sin explotarlo | `F.transform`, `F.filter`, `F.aggregate`: funciones de orden superior, reciben una lambda |

In [0]:
ejemplo = spark.createDataFrame(
    [("P1", [("Ceviche clásico", 2, 35.9), ("Chicha morada 1L", 1, 12.0)])],
    "id STRING, items ARRAY<STRUCT<producto: STRING, cantidad: INT, precio: DOUBLE>>",
)

# explode: una fila por item
display(ejemplo.select("id", F.explode("items").alias("item")).select("id", "item.*"))

# transform y filter: trabajan dentro del array, sin cambiar el número de filas
display(ejemplo.select(
    "id",
    F.transform("items", lambda x: F.upper(x["producto"])).alias("productos_mayus"),
    F.filter("items", lambda x: x["cantidad"] > 1).alias("items_con_varias_unidades"),
))

id,producto,cantidad,precio
P1,Ceviche clásico,2,35.9
P1,Chicha morada 1L,1,12.0


id,productos_mayus,items_con_varias_unidades
P1,"List(CEVICHE CLÁSICO, CHICHA MORADA 1L)","List(List(Ceviche clásico, 2, 35.9))"


### Ejercicio 4.1: pedidos válidos y únicos
A partir de `bronze_pedidos`, crea `pedidos_validos`:
- sin las filas corruptas
- sin pedidos repetidos: algunos pedidos se reenviaron y aparecen otra vez en el archivo del mes siguiente
- sin la columna `_corrupt_record`

Ojo: `distinct()` no sirve aquí. Las filas repetidas no son idénticas porque la columna `archivo` cambia. Hay que quitar duplicados por `id_pedido`.

In [0]:
from pyspark.sql.window import Window

ventana_pedidos = Window.partitionBy("id_pedido").orderBy(F.col("archivo").desc())
pedidos_validos = bronze_pedidos \
    .filter(F.col("_corrupt_record").isNull()) \
    .withColumn("rn", F.row_number().over(ventana_pedidos)) \
    .filter(F.col("rn") == 1) \
    .drop("_corrupt_record", "rn")
print(pedidos_validos.count())

11528


In [0]:
# comprobación
assert "_corrupt_record" not in pedidos_validos.columns
assert pedidos_validos.count() == 11528, f"Deberían quedar 11528 pedidos, hay {pedidos_validos.count()}"
assert pedidos_validos.select("id_pedido").distinct().count() == pedidos_validos.count()
ok()

Correcto


### Ejercicio 4.2: aplanar y tipar
A partir de `pedidos_validos`, crea `pedidos_planos` con estas columnas, en este orden:

| columna | de dónde sale |
|---|---|
| `id_pedido`, `id_cliente`, `id_restaurante`, `id_repartidor` | tal cual |
| `fecha_hora` | `fecha_hora` convertida a `TIMESTAMP` |
| `fecha` | la fecha (`DATE`) de `fecha_hora` |
| `hora` | la hora del día de `fecha_hora` |
| `anio_mes` | texto tipo `"2025-03"`, con `F.date_format(..., "yyyy-MM")` |
| `estado`, `metodo_pago`, `items` | tal cual |
| `n_items` | cantidad de elementos de `items` |
| `distancia_km` | `entrega.distancia_km` |
| `minutos_entrega` | `entrega.minutos` |
| `propina` | `entrega.propina`, con los nulos en 0 |
| `cupon`, `calificacion` | tal cual |

Las columnas `archivo` y `fecha_ingesta` eran de control de bronze, ya no van.

In [0]:
pedidos_planos = pedidos_validos.select(
    "id_pedido",
    "id_cliente",
    "id_restaurante",
    "id_repartidor",
    F.to_timestamp(F.col("fecha_hora")).alias("fecha_hora"),
    F.to_date(F.col("fecha_hora")).alias("fecha"),
    F.date_format(F.col("fecha_hora"), "HH:mm:ss").alias("hora"), # O la función de hora que corresponda según el formato deseado
    F.date_format(F.col("fecha_hora"), "yyyy-MM").alias("anio_mes"),
    "estado",
    "metodo_pago",
    "items",
    F.size(F.col("items")).alias("n_items"),
    F.col("entrega.distancia_km").alias("distancia_km"),
    F.col("entrega.minutos").alias("minutos_entrega"),
    F.coalesce(F.col("entrega.propina"), F.lit(0.0)).alias("propina"),
    "cupon",
    "calificacion"
)
display(pedidos_planos.limit(5))

id_pedido,id_cliente,id_restaurante,id_repartidor,fecha_hora,fecha,hora,anio_mes,estado,metodo_pago,items,n_items,distancia_km,minutos_entrega,propina,cupon,calificacion
CH-000001,1082,8,R01,2025-01-01T00:46:00.000Z,2025-01-01,00:46:00,2025-01,entregado,yape,"List(List(Bowl de quinua, 1, 27.9))",1,1.7,18,0.0,null,5
CH-000002,1398,7,R04,2025-01-01T04:20:22.000Z,2025-01-01,04:20:22,2025-01,entregado,yape,"List(List(Causa limeña, 2, 19.9), List(Arroz con pollo, 1, 26.9))",2,1.2,30,2.0,null,4
CH-000003,910,22,R02,2025-01-01T06:38:42.000Z,2025-01-01,06:38:42,2025-01,entregado,yape,"List(List(Maki furai, 2, 29.9), List(Nigiri mixto, 1, 36.9))",2,2.9,31,0.0,null,4
CH-000004,120,10,R05,2025-01-01T09:21:03.000Z,2025-01-01,09:21:03,2025-01,entregado,tarjeta,"List(List(Wantán frito, 1, 14.9), List(Tallarín saltado, 1, 26.9))",2,2.3,31,0.0,null,4
CH-000005,891,4,R04,2025-01-01T09:58:33.000Z,2025-01-01,09:58:33,2025-01,entregado,efectivo,"List(List(Pizza americana familiar, 1, 45.9))",1,2.9,20,1.0,null,5


In [0]:
# comprobación
esperadas = ["id_pedido", "id_cliente", "id_restaurante", "id_repartidor", "fecha_hora", "fecha", "hora", "anio_mes",
             "estado", "metodo_pago", "items", "n_items", "distancia_km", "minutos_entrega", "propina", "cupon", "calificacion"]
assert pedidos_planos.columns == esperadas, f"Columnas: {pedidos_planos.columns}"
tipos = dict(pedidos_planos.dtypes)
assert tipos["fecha_hora"] == "timestamp" and tipos["fecha"] == "date" and tipos["n_items"] == "int", f"Tipos: {tipos}"
assert pedidos_planos.filter(F.col("propina").isNull()).count() == 0, "Rellena las propinas nulas con 0"
meses = [r["anio_mes"] for r in pedidos_planos.select("anio_mes").distinct().orderBy("anio_mes").collect()]
assert meses == ["2025-01", "2025-02", "2025-03", "2025-04", "2025-05", "2025-06"], f"Meses: {meses}"
ok()

Correcto


### Ejercicio 4.3: una fila por producto (`explode`)
Crea `items_pedido` a partir de `pedidos_planos`, con una fila por cada producto de cada pedido y estas columnas:
`id_pedido`, `estado`, `id_restaurante`, `producto`, `cantidad`, `precio_unitario` y `subtotal_item` (= `cantidad * precio_unitario`, redondeado a 2).

In [0]:
items_pedido = pedidos_planos \
    .withColumn("item", F.explode("items")) \
    .select(
        "id_pedido",
        "estado",
        "id_restaurante",
        F.col("item.producto").alias("producto"),
        F.col("item.cantidad").alias("cantidad"),
        F.col("item.precio_unitario").alias("precio_unitario"),
        F.round(F.col("item.cantidad") * F.col("item.precio_unitario"), 2).alias("subtotal_item")
    )
display(items_pedido.limit(10))

id_pedido,estado,id_restaurante,producto,cantidad,precio_unitario,subtotal_item
CH-000001,entregado,8,Bowl de quinua,1,27.9,27.9
CH-000002,entregado,7,Causa limeña,2,19.9,39.8
CH-000002,entregado,7,Arroz con pollo,1,26.9,26.9
CH-000003,entregado,22,Maki furai,2,29.9,59.8
CH-000003,entregado,22,Nigiri mixto,1,36.9,36.9
CH-000004,entregado,10,Wantán frito,1,14.9,14.9
CH-000004,entregado,10,Tallarín saltado,1,26.9,26.9
CH-000005,entregado,4,Pizza americana familiar,1,45.9,45.9
CH-000006,entregado,29,Hamburguesa royal,1,24.9,24.9
CH-000006,entregado,29,Hamburguesa clásica,1,19.9,19.9


In [0]:
# comprobación
assert items_pedido.columns == ["id_pedido", "estado", "id_restaurante", "producto", "cantidad", "precio_unitario", "subtotal_item"], \
    f"Columnas: {items_pedido.columns}"
assert items_pedido.count() == pedidos_planos.agg(F.sum("n_items")).first()[0], "Debe haber una fila por cada item"
ok()

Correcto


### Ejercicio 4.4: subtotal sin explotar (`aggregate`)
Crea `pedidos_sub` agregando a `pedidos_planos` la columna `subtotal`: la suma de `cantidad * precio_unitario` de todos sus items, redondeada a 2.

Hazlo con `F.aggregate`, que recorre el array acumulando un valor, sin pasar por `explode` y `groupBy`:

```python
F.aggregate("items", F.lit(0.0), lambda acc, x: acc + x["cantidad"] * x["precio_unitario"])
```
El valor inicial es `0.0` (y no `0`) para que el acumulador sea `double` desde el principio.

In [0]:
pedidos_sub = pedidos_planos.withColumn(
    "subtotal",
    F.round(
        F.aggregate(
            "items", 
            F.lit(0.0), 
            lambda acc, x: acc + x["cantidad"] * x["precio_unitario"]
        ), 
        2
    )
)
display(pedidos_sub.select("id_pedido", "items", "subtotal").limit(5))

id_pedido,items,subtotal
CH-000001,"List(List(Bowl de quinua, 1, 27.9))",27.9
CH-000002,"List(List(Causa limeña, 2, 19.9), List(Arroz con pollo, 1, 26.9))",66.7
CH-000003,"List(List(Maki furai, 2, 29.9), List(Nigiri mixto, 1, 36.9))",96.7
CH-000004,"List(List(Wantán frito, 1, 14.9), List(Tallarín saltado, 1, 26.9))",41.8
CH-000005,"List(List(Pizza americana familiar, 1, 45.9))",45.9


In [0]:
# comprobación
por_items = items_pedido.groupBy("id_pedido").agg(F.sum("subtotal_item").alias("suma_items"))
dif = pedidos_sub.join(por_items, "id_pedido").filter(F.abs(F.col("subtotal") - F.col("suma_items")) > 0.011).count()
assert "subtotal" in pedidos_sub.columns and dif == 0, f"{dif} pedidos con un subtotal distinto a la suma de sus items"
ok("Correcto. Mismo resultado que explode + groupBy, pero sin multiplicar filas")

Correcto. Mismo resultado que explode + groupBy, pero sin multiplicar filas


### Ejercicio 4.5: aplicar cupones (left join + `when`)
A partir de `pedidos_sub`, crea `pedidos_total`:
1. Haz un left join con `F.broadcast(cupones)` donde `cupon == codigo`. Es left porque la mayoría de pedidos no tiene cupón, y algunos traen cupones que ya no existen (como `VERANO2024`).
2. Calcula `descuento`, redondeado a 2:
   - `0` si no hay cupón válido o si `subtotal < minimo_compra`
   - `subtotal * valor / 100` si `tipo == "porcentaje"`
   - `valor` si `tipo == "monto"`
3. Calcula `total = subtotal - descuento`, redondeado a 2.
4. Quita las columnas que vinieron de `cupones` (`codigo`, `tipo`, `valor`, `minimo_compra`).

In [0]:
pedidos_total = pedidos_sub.join(
    F.broadcast(cupones),
    pedidos_sub["cupon"] == cupones["codigo"],
    "left"
).withColumn(
    "descuento",
    F.round(
        F.when(
            (F.col("codigo").isNull()) | (F.col("subtotal") < F.col("minimo_compra")),
            0.0
        )
        .when(F.col("tipo") == "porcentaje", F.col("subtotal") * F.col("valor") / 100)
        .when(F.col("tipo") == "monto", F.col("valor"))
        .otherwise(0.0),
        2
    )
).withColumn(
    "total",
    F.round(F.col("subtotal") - F.col("descuento"), 2)
).drop("codigo", "tipo", "valor", "minimo_compra")
display(pedidos_total.filter(F.col("cupon").isNotNull()).select("id_pedido", "cupon", "subtotal", "descuento", "total").limit(10))

id_pedido,cupon,subtotal,descuento,total
CH-000006,CHASKI10,110.5,11.05,99.45
CH-000014,CHASKI10,9.9,0.99,8.91
CH-000019,FINDE15,38.9,0.0,38.9
CH-000026,CHASKI10,10.0,1.0,9.0
CH-000031,CHASKI10,10.0,1.0,9.0
CH-000035,CHASKI10,18.9,1.89,17.01
CH-000036,YAPEA5,42.9,5.0,37.9
CH-000053,FINDE15,24.9,0.0,24.9
CH-000055,FINDE15,104.6,15.69,88.91
CH-000058,YAPEA5,125.7,5.0,120.7


In [0]:
# comprobación
assert pedidos_total.count() == pedidos_sub.count(), "El left join no debe perder ni duplicar pedidos"
assert set(pedidos_total.columns) == set(pedidos_sub.columns) | {"descuento", "total"}, "Quita las columnas de cupones"
assert pedidos_total.filter(F.col("descuento").isNull() | F.col("total").isNull()).count() == 0, "No debería haber nulos"
assert pedidos_total.filter((F.col("cupon") == "VERANO2024") & (F.col("descuento") != 0)).count() == 0, "VERANO2024 no existe"
assert pedidos_total.filter((F.col("cupon") == "FINDE15") & (F.col("subtotal") < 40) & (F.col("descuento") != 0)).count() == 0, \
    "Revisa el mínimo de compra"
assert pedidos_total.filter((F.col("cupon") == "YAPEA5") & (F.col("subtotal") >= 25) & (F.col("descuento") != 5)).count() == 0, \
    "Revisa los cupones de tipo monto"
assert pedidos_total.filter((F.col("cupon") == "BIENVENIDA20") & (F.col("subtotal") >= 30) &
                            (F.abs(F.col("descuento") - F.col("subtotal") * 0.2) > 0.011)).count() == 0, \
    "Revisa los cupones de tipo porcentaje"
assert pedidos_total.filter(F.abs(F.col("total") - (F.col("subtotal") - F.col("descuento"))) > 0.011).count() == 0
ok()

Correcto


### Ejercicio 4.6: guardar `silver_pedidos`
Guarda `pedidos_total` como la tabla `silver_pedidos` y léela de nuevo en la variable `silver_pedidos`.

In [0]:
pedidos_total.write \
             .format("delta") \
             .mode("overwrite") \
             .option("overwriteSchema", "true") \
             .saveAsTable("silver_pedidos")

silver_pedidos = spark.table("silver_pedidos")
display(silver_pedidos.limit(10))

id_pedido,id_cliente,id_restaurante,id_repartidor,fecha_hora,fecha,hora,anio_mes,estado,metodo_pago,items,n_items,distancia_km,minutos_entrega,propina,cupon,calificacion,subtotal,descuento,total
CH-000001,1082,8,R01,2025-01-01T00:46:00.000Z,2025-01-01,00:46:00,2025-01,entregado,yape,"List(List(Bowl de quinua, 1, 27.9))",1,1.7,18,0.0,null,5,27.9,0.0,27.9
CH-000002,1398,7,R04,2025-01-01T04:20:22.000Z,2025-01-01,04:20:22,2025-01,entregado,yape,"List(List(Causa limeña, 2, 19.9), List(Arroz con pollo, 1, 26.9))",2,1.2,30,2.0,null,4,66.7,0.0,66.7
CH-000003,910,22,R02,2025-01-01T06:38:42.000Z,2025-01-01,06:38:42,2025-01,entregado,yape,"List(List(Maki furai, 2, 29.9), List(Nigiri mixto, 1, 36.9))",2,2.9,31,0.0,null,4,96.7,0.0,96.7
CH-000004,120,10,R05,2025-01-01T09:21:03.000Z,2025-01-01,09:21:03,2025-01,entregado,tarjeta,"List(List(Wantán frito, 1, 14.9), List(Tallarín saltado, 1, 26.9))",2,2.3,31,0.0,null,4,41.8,0.0,41.8
CH-000005,891,4,R04,2025-01-01T09:58:33.000Z,2025-01-01,09:58:33,2025-01,entregado,efectivo,"List(List(Pizza americana familiar, 1, 45.9))",1,2.9,20,1.0,null,5,45.9,0.0,45.9
CH-000006,925,29,R07,2025-01-01T10:59:47.000Z,2025-01-01,10:59:47,2025-01,entregado,yape,"List(List(Hamburguesa royal, 1, 24.9), List(Hamburguesa clásica, 1, 19.9), List(Papas fritas, 1, 9.9), List(Hamburguesa a lo pobre, 2, 27.9))",4,2.0,21,0.0,CHASKI10,null,110.5,11.05,99.45
CH-000007,526,3,R06,2025-01-01T11:12:38.000Z,2025-01-01,11:12:38,2025-01,entregado,tarjeta,"List(List(Arroz con mariscos, 1, 38.9))",1,2.0,12,0.0,null,4,38.9,0.0,38.9
CH-000008,178,26,null,2025-01-01T11:14:38.000Z,2025-01-01,11:14:38,2025-01,cancelado,efectivo,"List(List(Arroz chaufa, 1, 24.9))",1,2.3,null,0.0,null,null,24.9,0.0,24.9
CH-000009,359,19,null,2025-01-01T11:28:02.000Z,2025-01-01,11:28:02,2025-01,cancelado,yape,"List(List(Leche de tigre, 3, 18.9), List(Arroz con mariscos, 1, 38.9))",2,2.4,null,0.0,null,null,95.6,0.0,95.6
CH-000010,744,14,R04,2025-01-01T12:12:59.000Z,2025-01-01,12:12:59,2025-01,entregado,tarjeta,"List(List(Acevichado roll, 2, 32.9), List(Nigiri mixto, 1, 36.9), List(Maki furai, 1, 29.9), List(Gyozas, 1, 18.9))",4,1.5,15,5.0,null,5,151.5,0.0,151.5


In [0]:
# comprobación
assert silver_pedidos.count() == 11528
assert {"subtotal", "descuento", "total", "items"} <= set(silver_pedidos.columns)
ok()

Correcto


Hasta el 4.5 no se ejecutó nada: cada paso solo agregaba transformaciones al plan (evaluación perezosa). Recién el `saveAsTable` del 4.6 disparó todo. Con `explain()` puedes ver el plan físico que armó Spark; busca el `BroadcastHashJoin` del cruce con cupones:

In [0]:
pedidos_total.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   Project [id_pedido#14118, id_cliente#14119, id_restaurante#14120, id_repartidor#14121, fecha_hora#14185, fecha#14186, hora#14187, anio_mes#14188, estado#14123, metodo_pago#14124, items#14125, n_items#14189, distancia_km#14190, minutos_entrega#14191, propina#14192, cupon#14127, calificacion#14128, subtotal#14303, descuento#14414, round((subtotal#14303 - descuento#14414), 2) AS total#14416]
   +- Project [id_pedido#14118, id_cliente#14119, id_restaurante#14120, id_repartidor#14121, fecha_hora#14185, fecha#14186, hora#14187, anio_mes#14188, estado#14123, metodo_pago#14124, items#14125, n_items#14189, distancia_km#14190, minutos_entrega#14191, propina#14192, cupon#14127, calificacion#14128, subtotal#14303, round(CASE WHEN (isnull(codigo#14409) OR (subtotal#14303 < minimo_compra#14412)) THEN 0.0 WHEN (tipo#14410 = porcentaje) THEN ((subtotal#14303 * valor#14411) / 100.0) WHEN (tipo#14410 = monto) THEN valor#144

## Módulo 5. Joins con más nivel

Repaso de la práctica 1 y dos tipos nuevos:

| `how=` | Qué devuelve |
|---|---|
| `"inner"` | solo las filas con pareja en ambos lados |
| `"left"` | todas las filas de la izquierda, con `null` donde no hay pareja |
| `"left_anti"` | las filas de la izquierda que no tienen pareja |
| `"left_semi"` (nuevo) | las filas de la izquierda que sí tienen pareja, solo con las columnas de la izquierda y sin duplicarlas (como un `WHERE EXISTS` en SQL) |

Join por rango (nuevo): la condición no tiene que ser una igualdad. Por ejemplo, para asignar a cada pedido el tramo de distancia que le corresponde:

```python
df.join(tramos, (df.km >= tramos.desde) & (df.km < tramos.hasta))
```

Cuidado con los nombres repetidos: `silver_clientes` y `silver_restaurantes` tienen las dos columnas `nombre` y `ciudad`. Si las unes sin más, el resultado tiene dos columnas `ciudad` y Spark se queja de ambigüedad cuando intentas usarlas. Lo más limpio es renombrar antes del join, con `select` + `alias` o con `withColumnRenamed`.

### Ejercicio 5.1: pedidos huérfanos
Algunos pedidos tienen un `id_cliente` que no existe en `silver_clientes` (un problema de integridad que hay que reportar). Crea `huerfanos` con esos pedidos usando `left_anti` y guarda cuántos son en `n_huerfanos`.

In [0]:
huerfanos = silver_pedidos.join(
    silver_clientes,
    silver_pedidos["id_cliente"] == silver_clientes["id_cliente"],
    "left_anti"
)

n_huerfanos = huerfanos.count()
print(f"{n_huerfanos} pedidos de clientes que no existen")
display(huerfanos.groupBy("id_cliente").count().orderBy("id_cliente"))

80 pedidos de clientes que no existen


id_cliente,count
9001,3
9004,2
9005,3
9006,1
9007,1
9009,4
9010,3
9012,1
9013,4
9014,6


In [0]:
# comprobación
assert n_huerfanos == 80, f"Esperaba 80 pedidos huérfanos, tienes {n_huerfanos}"
assert set(huerfanos.columns) == set(silver_pedidos.columns), "left_anti solo devuelve las columnas de la izquierda"
ok()

Correcto


### Ejercicio 5.2: clientes activos e inactivos
Crea:
- `clientes_activos`: los clientes de `silver_clientes` que hicieron al menos un pedido (usa `left_semi`)
- `clientes_inactivos`: los que nunca pidieron (usa `left_anti`)

In [0]:
clientes_activos = silver_clientes.join(
    silver_pedidos,
    silver_clientes["id_cliente"] == silver_pedidos["id_cliente"],
    "left_semi"
)

clientes_inactivos = silver_clientes.join(
    silver_pedidos,
    silver_clientes["id_cliente"] == silver_pedidos["id_cliente"],
    "left_anti"
)
print(f"Activos: {clientes_activos.count()}  |  Sin pedidos: {clientes_inactivos.count()}")

Activos: 1406  |  Sin pedidos: 94


In [0]:
# comprobación
assert clientes_activos.columns == silver_clientes.columns, "left_semi solo devuelve las columnas de la izquierda"
assert clientes_activos.count() == silver_pedidos.join(silver_clientes, "id_cliente").select("id_cliente").distinct().count(), \
    "left_semi no repite clientes aunque tengan muchos pedidos"
assert clientes_activos.count() + clientes_inactivos.count() == silver_clientes.count()
ok()

Correcto


### Ejercicio 5.3: la tabla enriquecida
Crea `pedidos_enriquecidos` con todas las columnas de `silver_pedidos` más estas:

| columna nueva | viene de |
|---|---|
| `restaurante` | `silver_restaurantes.nombre` |
| `categoria`, `ciudad`, `comision_pct` | `silver_restaurantes` (la ciudad del restaurante) |
| `plan` | `silver_clientes.plan` |
| `ciudad_cliente` | `silver_clientes.ciudad` |

- Con restaurantes haz un inner join usando `F.broadcast` (es una tabla chica).
- Con clientes haz un left join, para no perder los pedidos huérfanos (van a quedar con `plan` nulo).
- Renombra antes de unir para que no haya columnas repetidas.

Este va a ser el DataFrame principal de los próximos módulos.

In [0]:
restaurantes_prep = silver_restaurantes.select(
    F.col("id_restaurante").alias("id_rest_join"),
    F.col("nombre").alias("restaurante"),
    "categoria",
    "ciudad",
    "comision_pct"
)
clientes_prep = silver_clientes.select(
    F.col("id_cliente").alias("id_cli_join"),
    "plan",
    F.col("ciudad").alias("ciudad_cliente")
)
pedidos_enriquecidos = silver_pedidos.join(
    F.broadcast(restaurantes_prep),
    silver_pedidos["id_restaurante"] == restaurantes_prep["id_rest_join"],
    "inner"
).drop("id_rest_join").join(
    clientes_prep,
    silver_pedidos["id_cliente"] == clientes_prep["id_cli_join"],
    "left"
).drop("id_cli_join")
display(pedidos_enriquecidos.select("id_pedido", "restaurante", "categoria", "ciudad", "plan", "ciudad_cliente", "total").limit(10))

id_pedido,restaurante,categoria,ciudad,plan,ciudad_cliente,total
CH-000001,Raíz Bowl,Vegetariana,Lima,free,Lima,27.9
CH-000002,La Sazón de Mamá Rosa,Criolla,Lima,prime,Lima,66.7
CH-000003,Kaizen Sushi,Sushi,Lima,free,Lima,96.7
CH-000004,Chifa Loto Azul,Chifa,Lima,prime,Lima,41.8
CH-000005,Pizza Nostra,Pizzería,Lima,prime,Lima,45.9
CH-000006,La Parrilla Urbana,Hamburguesas,Arequipa,free,Arequipa,99.45
CH-000007,Mar Picante,Cevichería,Lima,free,Lima,38.9
CH-000008,Chifa Dragón de Oro,Chifa,Arequipa,prime,Arequipa,24.9
CH-000009,Caleta Sabrosa,Cevichería,Lima,prime,Lima,95.6
CH-000010,Nikkei Roll,Sushi,Lima,free,Lima,151.5


In [0]:
# comprobación
assert pedidos_enriquecidos.count() == silver_pedidos.count(), "No deberías perder ni duplicar pedidos"
assert len(pedidos_enriquecidos.columns) == len(set(pedidos_enriquecidos.columns)), "Hay columnas repetidas: renombra antes del join"
assert {"restaurante", "categoria", "ciudad", "comision_pct", "plan", "ciudad_cliente"} <= set(pedidos_enriquecidos.columns)
assert set(silver_pedidos.columns) <= set(pedidos_enriquecidos.columns)
assert pedidos_enriquecidos.filter(F.col("plan").isNull()).count() == 80, "Los pedidos huérfanos deben quedar con plan nulo (left join)"
ok()

Correcto


### Ejercicio 5.4: costo de envío con un join por rango
El costo de envío depende de la distancia, según la tabla `tarifas_envio`: un pedido de 3.4 km cae en el tramo `[2, 5)` y paga 5 soles.

1. Crea `pedidos_envio` uniendo `pedidos_enriquecidos` con `tarifas_envio` donde `km_desde <= distancia_km < km_hasta`. Quita `km_desde` y `km_hasta` del resultado.
2. Crea `envio_por_ciudad` con los pedidos entregados: por `ciudad`, el número de pedidos (`n_pedidos`) y la suma de `costo_envio` (`ingreso_envios`, redondeada a 2). Ordena de mayor a menor `ingreso_envios`.

In [0]:
tarifas_envio = spark.createDataFrame(
    [(0.0, 2.0, 3.0), (2.0, 5.0, 5.0), (5.0, 8.0, 7.5), (8.0, 100.0, 10.0)],
    "km_desde DOUBLE, km_hasta DOUBLE, costo_envio DOUBLE",
)
pedidos_envio = pedidos_enriquecidos.join(
    tarifas_envio,
    (pedidos_enriquecidos["distancia_km"] >= tarifas_envio["km_desde"]) & 
    (pedidos_enriquecidos["distancia_km"] < tarifas_envio["km_hasta"]),
    "inner"
).drop("km_desde", "km_hasta")

envio_por_ciudad = pedidos_envio \
    .filter(F.col("estado") == "entregado") \
    .groupBy("ciudad") \
    .agg(
        F.count("id_pedido").alias("n_pedidos"),
        F.round(F.sum("costo_envio"), 2).alias("ingreso_envios")
    ) \
    .orderBy(F.col("ingreso_envios").desc())

display(envio_por_ciudad)

ciudad,n_pedidos,ingreso_envios
Lima,6519,31633.0
Arequipa,1909,9196.5
Trujillo,1718,8388.0


In [0]:
# comprobación
assert pedidos_envio.count() == pedidos_enriquecidos.count(), "Cada pedido debe caer en exactamente un tramo"
assert "km_desde" not in pedidos_envio.columns and "costo_envio" in pedidos_envio.columns
assert pedidos_envio.filter((F.col("distancia_km") < 2) & (F.col("costo_envio") != 3.0)).count() == 0
assert pedidos_envio.filter((F.col("distancia_km") == 5.0) & (F.col("costo_envio") != 7.5)).count() == 0, "5 km ya es del tramo [5, 8)"
assert envio_por_ciudad.columns == ["ciudad", "n_pedidos", "ingreso_envios"] and envio_por_ciudad.count() == 3
ok()

Correcto


## Módulo 6. Agregaciones con más nivel

Además de `count`, `sum`, `avg`, `min` y `max`:

```python
df.groupBy("grupo").agg(
    F.countDistinct("cliente").alias("clientes"),               # valores distintos
    F.median("monto").alias("mediana"),                         # mediana
    F.avg(F.when(F.col("estado") == "cancelado", 1).otherwise(0)).alias("tasa"),   # proporción que cumple algo
    F.avg(F.when(F.col("estado") == "entregado", F.col("monto"))).alias("media_entregados"),  # promedio de un subconjunto
    F.array_sort(F.collect_set("categoria")).alias("categorias"),   # junta los valores distintos en un array
)
```

El truco de `F.when(...)` sin `otherwise` dentro de una agregación es muy útil: las filas que no cumplen quedan en `null`, y `avg`, `sum`, `median` y compañía ignoran los nulos. Así puedes calcular varias métricas sobre subconjuntos distintos en un solo `groupBy`.

`pivot` convierte los valores de una columna en columnas:

```python
df.groupBy("mes").pivot("metodo_pago", ["yape", "plin"]).agg(F.sum("total"))
```
Si le pasas la lista de valores, Spark no tiene que recorrer los datos para averiguarlos y las columnas salen en ese orden.

### Ejercicio 6.1: KPIs por ciudad
Crea `kpis_ciudad` a partir de `pedidos_enriquecidos`, agrupando por `ciudad` (la del restaurante), con:
- `n_pedidos`: todos los pedidos
- `clientes_unicos`: clientes distintos
- `tasa_cancelacion`: porcentaje (0 a 100) de pedidos con estado `"cancelado"`, redondeado a 2
- `ticket_medio`: promedio de `total` solo de los pedidos entregados, redondeado a 2
- `ticket_mediano`: mediana de `total` solo de los pedidos entregados, redondeada a 2

Ordena por `n_pedidos` de mayor a menor. Todo en un solo `groupBy`.

In [0]:

kpis_ciudad = pedidos_enriquecidos.groupBy("ciudad").agg(
    F.count("id_pedido").alias("n_pedidos"),
    F.countDistinct("id_cliente").alias("clientes_unicos"),
    F.round(
        F.avg(F.when(F.col("estado") == "cancelado", 1).otherwise(0)) * 100, 
        2
    ).alias("tasa_cancelacion"),
    F.round(
        F.avg(F.when(F.col("estado") == "entregado", F.col("total"))), 
        2
    ).alias("ticket_medio"),
    F.round(
        F.median(F.when(F.col("estado") == "entregado", F.col("total"))), 
        2
    ).alias("ticket_mediano")
).orderBy(F.col("n_pedidos").desc())
display(kpis_ciudad)

ciudad,n_pedidos,clientes_unicos,tasa_cancelacion,ticket_medio,ticket_mediano
Lima,7349,1010,8.41,69.55,57.8
Arequipa,2231,406,10.89,65.98,53.8
Trujillo,1948,309,9.34,67.25,55.8


In [0]:
# comprobación
assert kpis_ciudad.columns == ["ciudad", "n_pedidos", "clientes_unicos", "tasa_cancelacion", "ticket_medio", "ticket_mediano"]
k = {r["ciudad"]: r for r in kpis_ciudad.collect()}
assert set(k) == {"Lima", "Arequipa", "Trujillo"}
assert sum(r["n_pedidos"] for r in k.values()) == pedidos_enriquecidos.count()
lima = pedidos_enriquecidos.filter("ciudad = 'Lima'")
assert abs(k["Lima"]["ticket_medio"] - lima.filter("estado = 'entregado'").agg(F.avg("total")).first()[0]) < 0.01, \
    "ticket_medio debe considerar solo los pedidos entregados"
assert abs(k["Lima"]["tasa_cancelacion"] - 100 * lima.filter("estado = 'cancelado'").count() / lima.count()) < 0.01, \
    "tasa_cancelacion es un porcentaje entre 0 y 100"
ok()

Correcto


### Ejercicio 6.2: ingresos por mes y método de pago (`pivot`)
Crea `ventas_mes_pago` con los pedidos entregados: una fila por `anio_mes` y una columna por método de pago (`yape`, `plin`, `tarjeta`, `efectivo`, en ese orden) con la suma de `total` redondeada a 2. Ordena por `anio_mes`.

Después del `display`, prueba armar un gráfico de barras apiladas desde el botón + > Visualización.

In [0]:
ventas_mes_pago = pedidos_enriquecidos \
    .filter(F.col("estado") == "entregado") \
    .groupBy("anio_mes") \
    .pivot("metodo_pago", ["yape", "plin", "tarjeta", "efectivo"]) \
    .agg(F.round(F.sum("total"), 2)) \
    .orderBy("anio_mes")
display(ventas_mes_pago)

anio_mes,yape,plin,tarjeta,efectivo
2025-01,25036.19,8120.59,16339.92,12169.01
2025-02,29743.0,12545.88,25315.19,15963.6
2025-03,46384.32,16778.61,33393.35,21017.98
2025-04,53397.7,21919.13,35987.74,27296.98
2025-05,52595.57,21592.76,42375.73,28418.21
2025-06,54206.46,20465.87,45090.26,28740.28


Databricks visualization. Run in Databricks to view.

In [0]:
# comprobación
assert ventas_mes_pago.columns == ["anio_mes", "yape", "plin", "tarjeta", "efectivo"], f"Columnas: {ventas_mes_pago.columns}"
filas = ventas_mes_pago.collect()
assert len(filas) == 6 and [r["anio_mes"] for r in filas] == sorted(r["anio_mes"] for r in filas)
suma = sum(r[c] or 0 for r in filas for c in ["yape", "plin", "tarjeta", "efectivo"])
total = pedidos_enriquecidos.filter("estado = 'entregado'").agg(F.sum("total")).first()[0]
assert abs(suma - total) < 1, "La suma de todas las celdas debería ser el total vendido (solo entregados)"
ok()

Correcto


### Ejercicio 6.3: categorías que prueba cada cliente
Crea `categorias_cliente` con una fila por `id_cliente` y estas columnas:
- `categorias`: array ordenado con las categorías distintas que pidió (usa `collect_set` y `array_sort`)
- `n_categorias`: cuántas son (`F.size`)

Luego crea `exploradores` con los clientes que probaron 5 categorías o más, y guarda cuántos son en `n_exploradores`.

In [0]:
categorias_cliente = pedidos_enriquecidos.groupBy("id_cliente").agg(
    F.array_sort(F.collect_set("categoria")).alias("categorias"),
    F.size(F.array_sort(F.collect_set("categoria"))).alias("n_categorias")
)
exploradores = categorias_cliente.filter(F.col("n_categorias") >= 5)
n_exploradores = exploradores.count()
print(f"{n_exploradores} clientes probaron 5 categorías o más")
display(exploradores.orderBy(F.desc("n_categorias")).limit(10))

613 clientes probaron 5 categorías o más


id_cliente,categorias,n_categorias
734,"List(Cevichería, Chifa, Criolla, Hamburguesas, Pizzería, Pollería, Sushi, Vegetariana)",8
275,"List(Cevichería, Chifa, Criolla, Hamburguesas, Pizzería, Pollería, Sushi, Vegetariana)",8
1405,"List(Cevichería, Chifa, Criolla, Hamburguesas, Pizzería, Pollería, Sushi, Vegetariana)",8
459,"List(Cevichería, Chifa, Criolla, Hamburguesas, Pizzería, Pollería, Sushi, Vegetariana)",8
603,"List(Cevichería, Chifa, Criolla, Hamburguesas, Pizzería, Pollería, Sushi, Vegetariana)",8
662,"List(Cevichería, Chifa, Criolla, Hamburguesas, Pizzería, Pollería, Sushi, Vegetariana)",8
826,"List(Cevichería, Chifa, Criolla, Hamburguesas, Pizzería, Pollería, Sushi, Vegetariana)",8
1342,"List(Cevichería, Chifa, Criolla, Hamburguesas, Pizzería, Pollería, Sushi, Vegetariana)",8
640,"List(Cevichería, Chifa, Criolla, Hamburguesas, Pizzería, Pollería, Sushi, Vegetariana)",8
731,"List(Cevichería, Chifa, Criolla, Hamburguesas, Pizzería, Pollería, Sushi, Vegetariana)",8


In [0]:
# comprobación
assert categorias_cliente.columns == ["id_cliente", "categorias", "n_categorias"]
assert categorias_cliente.count() == pedidos_enriquecidos.select("id_cliente").distinct().count()
assert categorias_cliente.agg(F.max("n_categorias")).first()[0] <= 8, "Usa collect_set (sin repetidos), no collect_list"
fila = categorias_cliente.filter(F.col("n_categorias") >= 3).first()
assert list(fila["categorias"]) == sorted(fila["categorias"]), "Ordena el array con array_sort"
assert n_exploradores == categorias_cliente.filter("n_categorias >= 5").count()
ok()

Correcto


### Ejercicio 6.4: los 10 productos que más venden
Con `items_pedido` (solo pedidos entregados), crea `top_productos` agrupando por `producto`:
- `unidades`: suma de `cantidad`
- `ingresos`: suma de `subtotal_item`, redondeada a 2
- `n_restaurantes`: en cuántos restaurantes distintos se vendió

Quédate con los 10 de mayores ingresos, de mayor a menor.

In [0]:
top_productos = items_pedido \
    .filter(F.col("estado") == "entregado") \
    .groupBy("producto") \
    .agg(
        F.sum("cantidad").alias("unidades"),
        F.round(F.sum("subtotal_item"), 2).alias("ingresos"),
        F.countDistinct("id_restaurante").alias("n_restaurantes")
    ) \
    .orderBy(F.desc("ingresos")) \
    .limit(10)
display(top_productos)

producto,unidades,ingresos,n_restaurantes
Pizza pepperoni familiar,1165,58133.5,5
Pizza americana familiar,1090,50031.0,5
Pizza hawaiana mediana,1078,37622.2,5
Pollo entero a la brasa,523,36557.7,5
Jalea mixta,709,30416.1,5
Arroz con mariscos,691,26879.9,5
Bowl de quinua,953,26588.7,5
Ceviche clásico,679,24376.1,5
Chicharrón de pescado,695,23560.5,5
Hamburguesa de lentejas,940,23406.0,5


In [0]:
# comprobación
filas = top_productos.collect()
assert top_productos.columns == ["producto", "unidades", "ingresos", "n_restaurantes"]
assert len(filas) == 10 and all(filas[i]["ingresos"] >= filas[i + 1]["ingresos"] for i in range(9))
maximo = (items_pedido.filter("estado = 'entregado'").groupBy("producto")
          .agg(F.round(F.sum("subtotal_item"), 2).alias("i")).agg(F.max("i")).first()[0])
assert filas[0]["ingresos"] == maximo, "¿Filtraste solo los entregados?"
ok()

Correcto


## Módulo 7. Ventanas con más nivel

En la práctica 1 usamos `row_number`, `lag` y un acumulado. Ahora vamos a jugar con el marco de la ventana (qué filas mira cada cálculo):

| Ventana | Qué filas usa |
|---|---|
| `Window.partitionBy("g").orderBy("f").rowsBetween(Window.unboundedPreceding, Window.currentRow)` | desde la primera fila del grupo hasta la actual (acumulado) |
| `Window.orderBy("fecha").rowsBetween(-6, 0)` | la fila actual y las 6 anteriores (media móvil de 7 filas) |
| `Window.partitionBy("g")` (sin `orderBy`) | todo el grupo: sirve para totales por grupo y porcentajes |
| `Window.partitionBy()` (vacía) | toda la tabla |

Y las funciones de ranking se diferencian en cómo tratan los empates:

In [0]:
notas = spark.createDataFrame([("Ana", 18), ("Luis", 17), ("Marta", 17), ("Jorge", 15), ("Sofía", 12)], "alumno STRING, nota INT")
w = Window.orderBy(F.desc("nota"))

display(notas.select(
    "alumno", "nota",
    F.row_number().over(w).alias("row_number"),   # 1, 2, 3, 4, 5: no repite
    F.rank().over(w).alias("rank"),               # 1, 2, 2, 4, 5: repite y salta
    F.dense_rank().over(w).alias("dense_rank"),   # 1, 2, 2, 3, 4: repite sin saltar
    F.ntile(2).over(w).alias("ntile_2"),          # parte las filas en 2 grupos de tamaño parecido
))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


alumno,nota,row_number,rank,dense_rank,ntile_2
Ana,18,1,1,1,1
Luis,17,2,2,2,1
Marta,17,3,2,2,1
Jorge,15,4,4,3,2
Sofía,12,5,5,4,2


Spark avisa con un warning cuando una ventana no tiene `partitionBy`, porque tiene que llevar todas las filas a una sola partición. Con tablas chicas como estas no es problema.

### Ejercicio 7.1: top 3 restaurantes por ciudad
1. Crea `ventas_restaurante` con los pedidos entregados de `pedidos_enriquecidos`, agrupando por `ciudad`, `id_restaurante` y `restaurante`, con `ingresos` = suma de `total` redondeada a 2. Agrupa también por el id porque hay cadenas con el mismo nombre en distintas ciudades.
2. Crea `top3_restaurantes` con los 3 restaurantes de más ingresos de cada ciudad. Usa `dense_rank` en una columna `puesto`. Columnas: `ciudad`, `id_restaurante`, `restaurante`, `ingresos`, `puesto`, ordenadas por ciudad y puesto.

In [0]:
ventas_restaurante = pedidos_enriquecidos \
    .filter(F.col("estado") == "entregado") \
    .groupBy("ciudad", "id_restaurante", "restaurante") \
    .agg(F.round(F.sum("total"), 2).alias("ingresos"))
ventana_ciudad = Window.partitionBy("ciudad").orderBy(F.col("ingresos").desc())
top3_restaurantes = ventas_restaurante \
    .withColumn("puesto", F.dense_rank().over(ventana_ciudad)) \
    .filter(F.col("puesto") <= 3) \
    .select("ciudad", "id_restaurante", "restaurante", "ingresos", "puesto") \
    .orderBy("ciudad", "puesto")
display(top3_restaurantes)

ciudad,id_restaurante,restaurante,ingresos,puesto
Arequipa,28,La Masa Madre,32030.03,1
Arequipa,29,La Parrilla Urbana,15173.37,2
Arequipa,25,La Leña Dorada,14982.26,3
Lima,4,Pizza Nostra,71165.06,1
Lima,19,Caleta Sabrosa,35799.73,2
Lima,9,El Gallo Fino,29603.7,3
Trujillo,40,Raíz Bowl,29337.03,1
Trujillo,36,Pizza Nostra,25650.81,2
Trujillo,35,Mar Picante,17635.65,3


In [0]:
# comprobación
assert top3_restaurantes.columns == ["ciudad", "id_restaurante", "restaurante", "ingresos", "puesto"]
assert top3_restaurantes.agg(F.max("puesto")).first()[0] == 3
assert top3_restaurantes.select("ciudad").distinct().count() == 3
maximos = ventas_restaurante.groupBy("ciudad").agg(F.max("ingresos").alias("maximo"))
assert top3_restaurantes.filter("puesto = 1").join(maximos, "ciudad").filter(F.col("ingresos") != F.col("maximo")).count() == 0, \
    "El puesto 1 debe ser el de mayores ingresos de su ciudad"
ok()

Correcto


### Ejercicio 7.2: participación de cada restaurante en su ciudad
A partir de `ventas_restaurante`, crea `participacion` agregando `pct_ciudad`: el porcentaje que representan los ingresos del restaurante sobre el total de su ciudad, redondeado a 2.

Pista: el total de la ciudad es `F.sum("ingresos").over(Window.partitionBy("ciudad"))`. No hace falta un groupBy ni un join.

In [0]:
ventana_total_ciudad = Window.partitionBy("ciudad")

# 2. Creamos participacion calculando el porcentaje del restaurante sobre el total de la ciudad
participacion = ventas_restaurante.withColumn(
    "pct_ciudad",
    F.round(
        (F.col("ingresos") / F.sum("ingresos").over(ventana_total_ciudad)) * 100,
        2
    )
).orderBy("ciudad", F.desc("pct_ciudad"))
display(participacion.orderBy("ciudad", F.desc("pct_ciudad")))

ciudad,id_restaurante,restaurante,ingresos,pct_ciudad
Arequipa,28,La Masa Madre,32030.03,25.43
Arequipa,29,La Parrilla Urbana,15173.37,12.05
Arequipa,25,La Leña Dorada,14982.26,11.89
Arequipa,27,Ola Marina,13652.6,10.84
Arequipa,31,El Fogón Limeño,12654.69,10.05
Arequipa,26,Chifa Dragón de Oro,12010.33,9.54
Arequipa,33,Pollos Don Brasa,10494.18,8.33
Arequipa,32,Verde Quinua,10432.87,8.28
Arequipa,30,Nikkei Roll,4528.39,3.6
Lima,4,Pizza Nostra,71165.06,15.7


In [0]:
# comprobación
assert participacion.count() == ventas_restaurante.count(), "Una ventana no colapsa filas"
for r in participacion.groupBy("ciudad").agg(F.sum("pct_ciudad").alias("s")).collect():
    assert abs(r["s"] - 100) < 0.2, f"En {r['ciudad']} los porcentajes suman {r['s']}, deberían sumar 100"
ok()

Correcto


### Ejercicio 7.3: tendencia diaria
`diario` ya tiene los pedidos entregados por día. Crea `tendencia` agregando:
- `media_movil_7d`: promedio de `n_pedidos` del día y los 6 anteriores, redondeado a 1
- `var_semanal`: `n_pedidos` menos los `n_pedidos` de 7 días antes (con `F.lag(..., 7)`)

Ordena por fecha. Después prueba un gráfico de líneas con `fecha` en el eje X y `n_pedidos` y `media_movil_7d` en el eje Y: la media móvil suaviza los picos de los fines de semana.

In [0]:
diario = (silver_pedidos
    .filter(F.col("estado") == "entregado")
    .groupBy("fecha")
    .agg(F.count("*").alias("n_pedidos"), F.round(F.sum("total"), 2).alias("ingresos"))
)
ventana_movil = Window.orderBy("fecha").rowsBetween(-6, 0)

ventana_lag = Window.orderBy("fecha")

tendencia = diario.withColumn(
    "media_movil_7d",
    F.round(F.avg("n_pedidos").over(ventana_movil), 1)
).withColumn(
    "var_semanal",
    F.col("n_pedidos") - F.lag("n_pedidos", 7).over(ventana_lag)
).orderBy("fecha")

display(tendencia)

fecha,n_pedidos,ingresos,media_movil_7d,var_semanal
2025-01-01,31,2011.16,31.0,null
2025-01-02,36,2200.18,33.5,null
2025-01-03,29,1489.3,32.0,null
2025-01-04,40,2423.12,34.0,null
2025-01-05,37,2819.33,34.6,null
2025-01-06,20,1524.85,32.2,null
2025-01-07,21,1451.83,30.6,null
2025-01-08,18,1546.7,28.7,-13
2025-01-09,28,1831.22,27.6,-8
2025-01-10,45,3141.1,29.9,16


Databricks visualization. Run in Databricks to view.

In [0]:
# comprobación
f = tendencia.orderBy("fecha").collect()
assert abs(f[0]["media_movil_7d"] - f[0]["n_pedidos"]) < 0.051, "El primer día solo se promedia a sí mismo"
assert abs(f[10]["media_movil_7d"] - sum(x["n_pedidos"] for x in f[4:11]) / 7) < 0.051, \
    "La media móvil debe usar el día actual y los 6 anteriores"
assert all(x["var_semanal"] is None for x in f[:7]), "Los primeros 7 días no tienen semana anterior"
assert f[7]["var_semanal"] == f[7]["n_pedidos"] - f[0]["n_pedidos"]
ok()

Correcto


### Ejercicio 7.4: cada cuántos días pide un cliente
1. Crea `con_anterior` a partir de los pedidos entregados de `silver_pedidos`, agregando `dias_desde_anterior`: días entre la `fecha` del pedido y la del pedido anterior del mismo cliente (`F.datediff` + `F.lag`). Ordena la ventana por `fecha_hora` y, para desempatar, por `id_pedido`.
2. Crea `frecuencia` con una fila por cliente: `id_cliente`, `n_pedidos` y `promedio_dias` (promedio de `dias_desde_anterior`, redondeado a 1). Deja solo a los clientes con 2 pedidos o más.

In [0]:
ventana_cliente = Window.partitionBy("id_cliente").orderBy("fecha_hora", "id_pedido")

con_anterior = silver_pedidos \
    .filter(F.col("estado") == "entregado") \
    .withColumn(
        "dias_desde_anterior",
        F.datediff(F.col("fecha"), F.lag("fecha", 1).over(ventana_cliente))
    )

frecuencia = con_anterior \
    .groupBy("id_cliente") \
    .agg(
        F.count("id_pedido").alias("n_pedidos"),
        F.round(F.avg("dias_desde_anterior"), 1).alias("promedio_dias")
    ) \
    .filter(F.col("n_pedidos") >= 2)
display(frecuencia.orderBy("promedio_dias").limit(10))

id_cliente,n_pedidos,promedio_dias
605,2,0.0
848,2,0.0
9037,2,0.0
1080,2,0.0
272,2,1.0
105,2,1.0
1003,2,1.0
262,3,1.0
1021,2,1.0
439,2,1.0


In [0]:
# comprobación
clientes = silver_pedidos.filter("estado = 'entregado'").select("id_cliente").distinct().count()
assert con_anterior.filter(F.col("dias_desde_anterior").isNull()).count() == clientes, \
    "Solo el primer pedido de cada cliente debería tener dias_desde_anterior nulo: ¿particionaste por cliente?"
assert con_anterior.filter(F.col("dias_desde_anterior") < 0).count() == 0, "Hay días negativos: revisa el orden de la ventana"
assert frecuencia.columns == ["id_cliente", "n_pedidos", "promedio_dias"]
assert frecuencia.filter(F.col("promedio_dias").isNull() | (F.col("n_pedidos") < 2)).count() == 0
ok()

Correcto


### Ejercicio 7.5: cuartiles de gasto (`ntile`)
¿Qué parte de los ingresos viene del 25 % de clientes que más gasta?

1. Crea `gasto_cliente` con los pedidos entregados de clientes registrados (`plan` no nulo en `pedidos_enriquecidos`): por `id_cliente`, `gasto_total` (suma de `total`, redondeada a 2) y `cuartil` = `ntile(4)` ordenando por `gasto_total` de mayor a menor (el cuartil 1 es el que más gasta).
2. Crea `resumen_cuartiles` con una fila por `cuartil`: `n_clientes`, `gasto_medio` (redondeado a 2) y `pct_ingresos` (porcentaje del gasto total que aporta ese cuartil, redondeado a 2). Ordena por cuartil.

Pista para `pct_ingresos`: las ventanas se calculan después del `groupBy`, así que se pueden aplicar sobre un agregado: `F.sum(F.sum("gasto_total")).over(Window.partitionBy())` es el total de toda la tabla.

In [0]:
gasto_cliente = pedidos_enriquecidos \
    .filter((F.col("estado") == "entregado") & (F.col("plan").isNotNull())) \
    .groupBy("id_cliente") \
    .agg(F.round(F.sum("total"), 2).alias("gasto_total"))

ventana_cuartil = Window.orderBy(F.col("gasto_total").desc())
gasto_cliente = gasto_cliente.withColumn(
    "cuartil",
    F.ntile(4).over(ventana_cuartil)
)
ventana_global = Window.partitionBy()
resumen_cuartiles = gasto_cliente \
    .groupBy("cuartil") \
    .agg(
        F.count("id_cliente").alias("n_clientes"),
        F.round(F.avg("gasto_total"), 2).alias("gasto_medio"),
        F.round(
            (F.sum("gasto_total") / F.sum(F.sum("gasto_total")).over(ventana_global)) * 100,
            2
        ).alias("pct_ingresos")
    ) \
    .orderBy("cuartil")
display(resumen_cuartiles)

cuartil,n_clientes,gasto_medio,pct_ingresos
1,346,1156.5,57.97
2,346,498.42,24.98
3,346,252.57,12.66
4,346,87.42,4.38


In [0]:
# comprobación
f = resumen_cuartiles.collect()
assert resumen_cuartiles.columns == ["cuartil", "n_clientes", "gasto_medio", "pct_ingresos"] and len(f) == 4
assert gasto_cliente.count() == pedidos_enriquecidos.filter("estado = 'entregado' AND plan IS NOT NULL").select("id_cliente").distinct().count()
assert max(r["n_clientes"] for r in f) - min(r["n_clientes"] for r in f) <= 1, "ntile reparte en grupos del mismo tamaño"
assert all(f[i]["gasto_medio"] > f[i + 1]["gasto_medio"] for i in range(3)), "El cuartil 1 debe ser el de mayor gasto"
assert abs(sum(r["pct_ingresos"] for r in f) - 100) < 0.1, "Los porcentajes deben sumar 100"
ok(f"Correcto. El 25 % de clientes que más gasta aporta el {f[0]['pct_ingresos']} % de los ingresos")

Correcto. El 25 % de clientes que más gasta aporta el 57.97 % de los ingresos


## Módulo 8. Spark SQL con más nivel

Tres cosas nuevas:

1. CTE (`WITH`): pones nombre a una subconsulta y la usas como si fuera una tabla. Hace que las consultas largas se lean de arriba hacia abajo.
   ```sql
   WITH ventas AS (SELECT ... GROUP BY ...),
        ranking AS (SELECT *, ROW_NUMBER() OVER (...) AS pos FROM ventas)
   SELECT * FROM ranking WHERE pos = 1
   ```
2. Consultas con parámetros: `spark.sql("... WHERE ciudad = :ciudad", args={"ciudad": "Lima"})`. Nunca armes SQL pegando con f-strings un valor que escribió un usuario: alguien podría mandar `Lima' OR '1'='1` y cambiar el sentido de la consulta (inyección SQL). Con parámetros, el valor siempre se trata como dato.
3. Las tablas que guardaste (`silver_pedidos`, etc.) se pueden consultar directo por su nombre, sin crear vistas.

### Ejercicio 8.1: vista temporal + CTE + ventana en SQL
Registra `pedidos_enriquecidos` como la vista temporal `pedidos_enriquecidos`. Luego, con `spark.sql` y un CTE, crea `top_categoria_mes`: para cada `anio_mes`, la categoría con más ingresos (suma de `total` de los pedidos entregados, redondeada a 2). Columnas: `anio_mes`, `categoria`, `ingresos`, ordenado por `anio_mes`.

Cuando tengas el resultado, míralo con calma: ¿la categoría líder es la misma todo el semestre? (Pista: en Perú, de enero a marzo es verano.)

In [0]:
pedidos_enriquecidos.createOrReplaceTempView("pedidos_enriquecidos")
top_categoria_mes = spark.sql("""
    WITH ingresos_categoria_mes AS (
        SELECT 
            anio_mes,
            categoria,
            ROUND(SUM(total), 2) AS ingresos
        FROM pedidos_enriquecidos
        WHERE estado = 'entregado'
        GROUP BY anio_mes, categoria
    ),
    ranking_categorias AS (
        SELECT 
            anio_mes,
            categoria,
            ingresos,
            ROW_NUMBER() OVER (PARTITION BY anio_mes ORDER BY ingresos DESC) AS rk
        FROM ingresos_categoria_mes
    )
    SELECT 
        anio_mes,
        categoria,
        ingresos
    FROM ranking_categorias
    WHERE rk = 1
    ORDER BY anio_mes
""")
display(top_categoria_mes)

anio_mes,categoria,ingresos
2025-01,Cevichería,18201.5
2025-02,Cevichería,24130.83
2025-03,Cevichería,28079.04
2025-04,Pizzería,38326.12
2025-05,Pizzería,33927.72
2025-06,Pizzería,35338.81


In [0]:
# comprobación
assert top_categoria_mes.columns == ["anio_mes", "categoria", "ingresos"] and top_categoria_mes.count() == 6
esperado = (pedidos_enriquecidos.filter("estado = 'entregado'")
            .groupBy("anio_mes", "categoria").agg(F.round(F.sum("total"), 2).alias("ingresos"))
            .groupBy("anio_mes").agg(F.max("ingresos").alias("ingresos")))
assert {(r["anio_mes"], r["ingresos"]) for r in esperado.collect()} == \
       {(r["anio_mes"], r["ingresos"]) for r in top_categoria_mes.collect()}
ok()

Correcto


### Ejercicio 8.2: consulta con parámetros
Escribe la función `resumen_ciudad_mes(ciudad, anio_mes)`, que devuelva un DataFrame de una fila con:
- `n_pedidos`: todos los pedidos de esa ciudad y mes
- `ingresos`: suma de `total` solo de los entregados, redondeada a 2 (pista: `SUM(CASE WHEN ... THEN total END)`)
- `calificacion_media`: promedio de `calificacion`, redondeado a 1

Consulta la vista `pedidos_enriquecidos` con `spark.sql` usando los marcadores `:ciudad` y `:anio_mes` y el argumento `args`.

In [0]:
def resumen_ciudad_mes(ciudad, anio_mes):
    return spark.sql("""
            SELECT 
                COUNT(id_pedido) AS n_pedidos,
                ROUND(SUM(CASE WHEN estado = 'entregado' THEN total END), 2) AS ingresos,
                ROUND(AVG(calificacion), 1) AS calificacion_media
            FROM pedidos_enriquecidos
            WHERE ciudad = :ciudad AND anio_mes = :anio_mes
        """, args={"ciudad": ciudad, "anio_mes": anio_mes})


display(resumen_ciudad_mes("Arequipa", "2025-03"))

n_pedidos,ingresos,calificacion_media
364,20312.64,4.3


In [0]:
# comprobación
for ciudad, mes in [("Arequipa", "2025-03"), ("Lima", "2025-06")]:
    r = resumen_ciudad_mes(ciudad, mes).first()
    filtro = (F.col("ciudad") == ciudad) & (F.col("anio_mes") == mes)
    assert r["n_pedidos"] == pedidos_enriquecidos.filter(filtro).count(), f"n_pedidos no cuadra para {ciudad} {mes}"
    esperado = pedidos_enriquecidos.filter(filtro & (F.col("estado") == "entregado")).agg(F.sum("total")).first()[0]
    assert abs(r["ingresos"] - esperado) < 0.01, f"ingresos no cuadra para {ciudad} {mes}"
ok()

Correcto


### Ejercicio 8.3: celda `%sql` sobre una tabla
En la celda de abajo, consulta directamente la tabla `silver_pedidos` (solo pedidos entregados) y devuelve, para cada `hora`:
- `franja`: `"almuerzo"` si la hora está entre 12 y 15, `"cena"` si está entre 19 y 22, `"otra"` en los demás casos (`CASE WHEN`)
- `n_pedidos`
- `minutos_medios`: promedio de `minutos_entrega`, redondeado a 1

Ordena por hora y arma un gráfico de barras con el resultado.

Dato: Databricks SQL tiene además la cláusula `QUALIFY`, que filtra por el resultado de una ventana sin necesidad de subconsulta (por ejemplo `QUALIFY ROW_NUMBER() OVER (...) = 1`). No es parte del Spark estándar, pero en Databricks funciona y ahorra el CTE del 8.1.

In [0]:
%sql
-- consulta sobre silver_pedidos
SELECT 
    HOUR(fecha_hora) AS hora,
    CASE 
        WHEN HOUR(fecha_hora) BETWEEN 12 AND 15 THEN 'almuerzo'
        WHEN HOUR(fecha_hora) BETWEEN 19 AND 22 THEN 'cena'
        ELSE 'otra'
    END AS franja,
    COUNT(id_pedido) AS n_pedidos,
    ROUND(AVG(minutos_entrega), 1) AS minutos_medios
FROM silver_pedidos
WHERE estado = 'entregado'
GROUP BY 1, 2
ORDER BY hora;

hora,franja,n_pedidos,minutos_medios
0,otra,100,27.5
1,otra,54,25.8
2,otra,33,25.3
3,otra,26,26.5
4,otra,18,24.9
5,otra,26,23.1
6,otra,90,25.2
7,otra,166,25.2
8,otra,219,27.7
9,otra,265,26.0


## Módulo 9. Delta Lake: MERGE y time travel

Cada escritura en una tabla Delta crea una versión nueva. Eso permite:
- ver el historial de cambios: `DESCRIBE HISTORY tabla`
- leer una versión anterior (time travel): `SELECT * FROM tabla VERSION AS OF 3`, o en Python `spark.read.option("versionAsOf", 3).table("tabla")`
- volver atrás si algo salió mal: `RESTORE TABLE tabla TO VERSION AS OF 3`
- actualizar e insertar en una sola operación con `MERGE INTO` (lo que se suele llamar upsert):

```sql
MERGE INTO destino AS d
USING origen AS o
ON d.id = o.id
WHEN MATCHED THEN UPDATE SET *        -- si ya existe, se actualiza
WHEN NOT MATCHED THEN INSERT *        -- si no existe, se inserta
```
`UPDATE SET *` e `INSERT *` funcionan cuando origen y destino tienen las mismas columnas. Si no, se listan: `UPDATE SET d.comision_pct = o.comision_pct`.

Llegó el archivo de julio con cambios de restaurantes: algunos cambiaron su comisión y hay dos restaurantes nuevos. En lugar de reescribir toda la tabla, hacemos un MERGE.

### Ejercicio 9.1: preparar los cambios
1. Lee `ruta_cambios` con el mismo formato que `restaurantes.csv` (encabezado, `;` y `ESQUEMA_RESTAURANTES`) y límpialo con tu función `limpiar_restaurantes` usando `transform`. Guárdalo en `cambios_restaurantes`.
2. Guarda en `version_antes` el número de la versión actual de `silver_restaurantes`: es el máximo de la columna `version` de `DESCRIBE HISTORY silver_restaurantes`.

In [0]:
ruta_cambios = RUTA_RAW + "/actualizaciones/restaurantes_2025_07.csv"
cambios_restaurantes = (
    spark.read.option("header", "true")
    .option("sep", ";")
    .schema(ESQUEMA_RESTAURANTES)
    .csv(ruta_cambios)
    .transform(limpiar_restaurantes)
)
version_antes = (
    spark.sql("DESCRIBE HISTORY silver_restaurantes")
    .agg(F.max("version"))
    .collect()[0][0]
)

display(cambios_restaurantes)
print("Versión actual de silver_restaurantes:", version_antes)

id_restaurante,nombre,categoria,ciudad,distrito,comision_pct,fecha_alta
29,La Parrilla Urbana Express,Hamburguesas,Arequipa,Paucarpata,24.5,2020-09-03
21,Burger Chalaca,Hamburguesas,Lima,Surco,20.0,2024-09-08
8,Raíz Bowl,Vegetariana,Lima,La Molina,17.0,2024-02-13
35,Mar Picante,Cevichería,Trujillo,La Esperanza,9.0,2022-05-31
41,Kaizen Sushi,Sushi,Lima,Los Olivos,15.0,2025-07-01
42,Ola Marina,Cevichería,Arequipa,Cayma,15.0,2025-07-01


Versión actual de silver_restaurantes: 3


In [0]:
# comprobación
assert cambios_restaurantes.count() == 6
assert cambios_restaurantes.columns == silver_restaurantes.columns, "Usa limpiar_restaurantes para tener las mismas columnas que la tabla"
assert isinstance(version_antes, int) and version_antes >= 0
ok()

Correcto


### Ejercicio 9.2: `MERGE INTO`
Registra `cambios_restaurantes` como la vista temporal `cambios_restaurantes` y haz un `MERGE INTO silver_restaurantes` que actualice los restaurantes que ya existen (por `id_restaurante`) e inserte los nuevos. Muestra el resultado del `spark.sql` (trae cuántas filas se actualizaron e insertaron) y vuelve a leer la tabla en `silver_restaurantes`.

In [0]:
cambios_restaurantes.createOrReplaceTempView("cambios_restaurantes")
resultado_merge = spark.sql("""
    MERGE INTO silver_restaurantes AS d
    USING cambios_restaurantes AS o
    ON d.id_restaurante = o.id_restaurante
    WHEN MATCHED THEN 
        UPDATE SET *
    WHEN NOT MATCHED THEN 
        INSERT *
""")
display(resultado_merge)
silver_restaurantes = spark.table("silver_restaurantes")
display(silver_restaurantes.orderBy("id_restaurante"))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
6,4,0,2


id_restaurante,nombre,categoria,ciudad,distrito,comision_pct,fecha_alta
1,El Gallo Fino,Pollería,Lima,San Isidro,20.0,2020-12-10
2,Chifa Bambú,Chifa,Lima,Barranco,12.0,2023-04-17
3,Mar Picante,Cevichería,Lima,Los Olivos,12.0,2023-08-13
4,Pizza Nostra,Pizzería,Lima,Barranco,12.0,2021-07-22
5,Doble Queso,Hamburguesas,Lima,La Molina,12.0,2020-03-29
6,Nikkei Roll,Sushi,Lima,Miraflores,12.0,2020-07-11
7,La Sazón de Mamá Rosa,Criolla,Lima,San Miguel,18.0,2023-12-19
8,Raíz Bowl,Vegetariana,Lima,La Molina,17.0,2024-02-13
9,El Gallo Fino,Pollería,Lima,Los Olivos,22.5,2019-04-27
10,Chifa Loto Azul,Chifa,Lima,La Molina,22.5,2022-07-02


In [0]:
# comprobación
assert silver_restaurantes.count() == 42, "Deberían quedar 42 restaurantes (40 + 2 nuevos)"
nuevos = {r["id_restaurante"]: r["comision_pct"] for r in cambios_restaurantes.collect()}
actuales = {r["id_restaurante"]: r["comision_pct"] for r in silver_restaurantes.collect()}
assert all(actuales[k] == v for k, v in nuevos.items()), "Las comisiones no se actualizaron"
assert spark.sql("DESCRIBE HISTORY silver_restaurantes").agg(F.max("version")).first()[0] > version_antes
ok()

Correcto


El MERGE quedó registrado en el historial. En `operationMetrics` puedes ver cuántas filas se actualizaron (`numTargetRowsUpdated`) y cuántas se insertaron (`numTargetRowsInserted`):

In [0]:
display(spark.sql("DESCRIBE HISTORY silver_restaurantes").select("version", "timestamp", "operation", "operationMetrics"))

version,timestamp,operation,operationMetrics
4,2026-11-10T04:02:59.000Z,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 3330, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 4, executionTimeMs -> 4241, materializeSourceTimeMs -> 547, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1710, numTargetRowsUpdated -> 4, numOutputRows -> 6, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 6, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1903)"
3,2026-11-10T03:48:18.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 3120, numDeletionVectorsRemoved -> 0, numOutputRows -> 40, numOutputBytes -> 3120)"
2,2026-11-09T22:08:51.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 3120, numDeletionVectorsRemoved -> 0, numOutputRows -> 40, numOutputBytes -> 3120)"
1,2026-11-09T21:07:58.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 3120, numDeletionVectorsRemoved -> 0, numOutputRows -> 40, numOutputBytes -> 3120)"
0,2026-11-09T21:07:34.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 40, numOutputBytes -> 3120)"


### Ejercicio 9.3: time travel
Lee la versión anterior al MERGE con `VERSION AS OF version_antes` y crea `comparacion` con los restaurantes cuya comisión cambió. Columnas: `id_restaurante`, `nombre` (el de la versión anterior), `comision_antes` y `comision_despues`, ordenado por `id_restaurante`.

Pista: como vas a unir la tabla consigo misma (dos versiones), renombra `comision_pct` en cada lado antes del join.

In [0]:
silver_antigua = spark.read.option("versionAsOf", version_antes).table("silver_restaurantes")
silver_actual = spark.table("silver_restaurantes")
antigua_prep = silver_antigua.select(
    "id_restaurante",
    F.col("nombre"),
    F.col("comision_pct").alias("comision_antes")
)
actual_prep = silver_actual.select(
    "id_restaurante",
    F.col("comision_pct").alias("comision_despues")
)
comparacion = antigua_prep.join(
    actual_prep,
    "id_restaurante",
    "inner"
).filter(
    F.col("comision_antes") != F.col("comision_despues")
).select(
    "id_restaurante",
    "nombre",
    "comision_antes",
    "comision_despues"
).orderBy("id_restaurante")
display(comparacion)

id_restaurante,nombre,comision_antes,comision_despues
8,Raíz Bowl,20.0,17.0
21,Burger Chalaca,18.0,20.0
29,La Parrilla Urbana,22.5,24.5
35,Mar Picante,12.0,9.0


In [0]:
# comprobación
assert comparacion.columns == ["id_restaurante", "nombre", "comision_antes", "comision_despues"]
assert spark.sql(f"SELECT * FROM silver_restaurantes VERSION AS OF {version_antes}").count() == 40, \
    "version_antes debería ser la versión con 40 restaurantes. ¿Volviste a ejecutar el 9.1 después del MERGE?"
assert comparacion.count() == 4, f"Deberían ser 4 restaurantes con comisión distinta, hay {comparacion.count()}"
ok()

Correcto


## Módulo 10. Gold y exportación

La capa gold tiene tablas pensadas para quien consume los datos: dashboards, reportes o el equipo comercial. Además vamos a exportar archivos a un volumen:

- Parquet particionado: `df.write.partitionBy("anio_mes").parquet(ruta)` crea una carpeta por valor (`anio_mes=2025-01/`, `anio_mes=2025-02/`...). Al leer con un filtro sobre esa columna, Spark solo abre las carpetas que necesita (partition pruning).
- Un CSV en un solo archivo: por defecto Spark escribe un archivo por partición. Con `coalesce(1)` se junta todo en una partición y sale un único `part-*.csv`. Úsalo solo con resultados chicos, porque obliga a que una sola tarea escriba todo.

### Ejercicio 10.1: tabla gold de ventas diarias
Crea `gold_ventas_diarias_df` con los pedidos entregados de `pedidos_enriquecidos`, agrupando por `fecha` y `ciudad`:
- `n_pedidos`
- `ingresos`: suma de `total`, redondeada a 2
- `ticket_medio`: promedio de `total`, redondeado a 2
- `comision_chaskifood`: suma de `total * comision_pct / 100`, redondeada a 2 (lo que gana ChaskiFood)

Ordena por fecha y ciudad, guárdalo como la tabla `gold_ventas_diarias` y léela en `gold_ventas_diarias`.

Un detalle para pensar: como `silver_restaurantes` ya tiene las comisiones de julio, este cálculo aplica la comisión nueva también a pedidos de meses anteriores. Para hacerlo bien habría que guardar el historial de comisiones con sus fechas de vigencia (lo que se conoce como dimensión de cambio lento, SCD tipo 2). Lo dejamos así para no complicar el ejercicio.

In [0]:
gold_ventas_diarias_df = pedidos_enriquecidos \
    .filter(F.col("estado") == "entregado") \
    .groupBy("fecha", "ciudad") \
    .agg(
        F.count("id_pedido").alias("n_pedidos"),
        F.round(F.sum("total"), 2).alias("ingresos"),
        F.round(F.avg("total"), 2).alias("ticket_medio"),
        F.round(F.sum(F.col("total") * F.col("comision_pct") / 100), 2).alias("comision_chaskifood")
    ) \
    .orderBy("fecha", "ciudad")
gold_ventas_diarias_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_ventas_diarias")
gold_ventas_diarias = spark.table("gold_ventas_diarias")
display(gold_ventas_diarias.limit(10))

fecha,ciudad,n_pedidos,ingresos,ticket_medio,comision_chaskifood
2025-01-01,Arequipa,8,522.15,65.27,87.88
2025-01-01,Lima,18,1166.41,64.8,204.35
2025-01-01,Trujillo,5,322.6,64.52,37.55
2025-01-02,Arequipa,8,401.56,50.19,81.27
2025-01-02,Lima,22,1446.91,65.77,246.72
2025-01-02,Trujillo,6,351.71,58.62,49.43
2025-01-03,Arequipa,2,109.97,54.99,26.94
2025-01-03,Lima,19,949.71,49.98,163.43
2025-01-03,Trujillo,8,429.62,53.7,52.52
2025-01-04,Arequipa,8,361.3,45.16,58.65


In [0]:
# comprobación
assert gold_ventas_diarias.columns == ["fecha", "ciudad", "n_pedidos", "ingresos", "ticket_medio", "comision_chaskifood"]
assert gold_ventas_diarias.groupBy("fecha", "ciudad").count().filter("count > 1").count() == 0, "Debe haber una fila por fecha y ciudad"
assert gold_ventas_diarias.agg(F.sum("n_pedidos")).first()[0] == pedidos_enriquecidos.filter("estado = 'entregado'").count()
assert gold_ventas_diarias.filter(F.col("comision_chaskifood") >= F.col("ingresos")).count() == 0, "La comisión es un porcentaje de los ingresos"
ok()

Correcto


### Ejercicio 10.2: exportar a Parquet particionado
1. Escribe `pedidos_enriquecidos` en formato Parquet en `ruta_parquet`, particionado por `anio_mes`, en modo overwrite.
2. Crea `pedidos_marzo` leyendo esa carpeta y filtrando `anio_mes == "2025-03"`. Fíjate que `anio_mes` vuelve a aparecer como columna aunque no está dentro de los archivos: Spark la saca del nombre de la carpeta.

In [0]:
ruta_parquet = RUTA_VOLUMEN + "/export/pedidos_parquet"

pedidos_enriquecidos.write \
    .format("parquet") \
    .mode("overwrite") \
    .partitionBy("anio_mes") \
    .save(ruta_parquet)
pedidos_marzo = spark.read \
    .format("parquet") \
    .load(ruta_parquet) \
    .filter(F.col("anio_mes") == "2025-03")

display(dbutils.fs.ls(ruta_parquet))
print(pedidos_marzo.count())

path,name,size,modificationTime
dbfs:/Volumes/workspace/chaskifood/archivos/export/pedidos_parquet/_SUCCESS,_SUCCESS,0,1794284297000
dbfs:/Volumes/workspace/chaskifood/archivos/export/pedidos_parquet/anio_mes=2025-01/,anio_mes=2025-01/,0,1794284296979
dbfs:/Volumes/workspace/chaskifood/archivos/export/pedidos_parquet/anio_mes=2025-02/,anio_mes=2025-02/,0,1794284296979
dbfs:/Volumes/workspace/chaskifood/archivos/export/pedidos_parquet/anio_mes=2025-03/,anio_mes=2025-03/,0,1794284296980
dbfs:/Volumes/workspace/chaskifood/archivos/export/pedidos_parquet/anio_mes=2025-04/,anio_mes=2025-04/,0,1794284296980
dbfs:/Volumes/workspace/chaskifood/archivos/export/pedidos_parquet/anio_mes=2025-05/,anio_mes=2025-05/,0,1794284296980
dbfs:/Volumes/workspace/chaskifood/archivos/export/pedidos_parquet/anio_mes=2025-06/,anio_mes=2025-06/,0,1794284296980


1888


In [0]:
# comprobación
carpetas = sorted(f.name for f in dbutils.fs.ls(ruta_parquet) if f.name.startswith("anio_mes="))
assert len(carpetas) == 6, f"Esperaba 6 carpetas anio_mes=..., hay {carpetas}"
assert pedidos_marzo.count() == pedidos_enriquecidos.filter(F.col("anio_mes") == "2025-03").count()
ok()

Correcto


En el plan de `pedidos_marzo` aparece el filtro como `PartitionFilters`: Spark sabe que solo tiene que leer la carpeta `anio_mes=2025-03`.

In [0]:
pedidos_marzo.explain()

== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonScan parquet [id_pedido#23326,id_cliente#23327,id_restaurante#23328,id_repartidor#23329,fecha_hora#23330,fecha#23331,hora#23332,estado#23333,metodo_pago#23334,items#23335,n_items#23336,distancia_km#23337,minutos_entrega#23338,propina#23339,cupon#23340,calificacion#23341,subtotal#23342,descuento#23343,total#23344,restaurante#23345,categoria#23346,ciudad#23347,comision_pct#23348,plan#23349,ciudad_cliente#23350,... 1 more fields] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/workspace/chaskifood/archivos/export/pedidos_parquet], OptionalDataFilters: [], PartitionFilters: [isnotnull(anio_mes#23351), (anio_mes#23351 = 2025-03)], ReadSchema: struct<id_pedido:string,id_cliente:int,id_restaurante:int,id_repartidor:string,fecha_hora:timesta..., RequiredDataFilters: []


== Photon Explanation ==
The query is fully supported by Photon.


### Ejercicio 10.3: reporte CSV en un solo archivo
Escribe `top3_restaurantes` (ejercicio 7.1) como CSV con encabezado en `ruta_reporte`, en modo overwrite, usando `coalesce(1)` para que quede en un único archivo `part-*`.

In [0]:
ruta_reporte = RUTA_VOLUMEN + "/export/top3_restaurantes"

top3_restaurantes \
    .coalesce(1) \
    .write \
    .format("csv") \
    .option("header", "true") \
    .mode("overwrite") \
    .save(ruta_reporte)
display(dbutils.fs.ls(ruta_reporte))

path,name,size,modificationTime
dbfs:/Volumes/workspace/chaskifood/archivos/export/top3_restaurantes/_SUCCESS,_SUCCESS,0,1794284497000
dbfs:/Volumes/workspace/chaskifood/archivos/export/top3_restaurantes/_committed_2359524842938745995,_committed_2359524842938745995,113,1794284497000
dbfs:/Volumes/workspace/chaskifood/archivos/export/top3_restaurantes/_started_2359524842938745995,_started_2359524842938745995,0,1794284496000
dbfs:/Volumes/workspace/chaskifood/archivos/export/top3_restaurantes/part-00000-tid-2359524842938745995-25883d84-b9cb-4385-9fae-f4a0993bff2c-615-1-c000.csv,part-00000-tid-2359524842938745995-25883d84-b9cb-4385-9fae-f4a0993bff2c-615-1-c000.csv,377,1794284497000


In [0]:
# comprobación
partes = [f.name for f in dbutils.fs.ls(ruta_reporte) if f.name.startswith("part-")]
assert len(partes) == 1, f"Debería haber un solo archivo part-*, hay {len(partes)}"
assert spark.read.option("header", True).csv(ruta_reporte).count() == top3_restaurantes.count()
ok("Correcto. Desde Catalog > workspace > chaskifood > archivos puedes descargar el CSV")

Correcto. Desde Catalog > workspace > chaskifood > archivos puedes descargar el CSV


## Desafío final

Dos preguntas que el equipo de negocio de ChaskiFood hace siempre. No hay plantilla: decide tú los pasos.

### Desafío A: retención por cohortes
De los clientes que hicieron su primer pedido en cada mes, ¿qué porcentaje volvió a pedir 1, 2, 3... meses después?

Usa solo pedidos entregados de clientes registrados (`plan` no nulo en `pedidos_enriquecidos`). El resultado, `retencion`, debe tener:

| columna | significado |
|---|---|
| `mes_cohorte` | mes del primer pedido del cliente (`"2025-01"`, ...) |
| `clientes` | cuántos clientes empezaron ese mes |
| `m0` ... `m5` | porcentaje de esos clientes que pidió 0, 1, ..., 5 meses después de su primer mes (redondeado a 1). `m0` siempre es 100. Donde todavía no hay datos (por ejemplo, `m1` de la cohorte de junio) queda `null` |

Ordena por `mes_cohorte`.

Un camino posible:
1. `mes_cohorte` = el `anio_mes` mínimo de cada cliente (una ventana `partitionBy("id_cliente")` sin `orderBy`).
2. `meses_desde` = meses entre `anio_mes` y `mes_cohorte`: `F.months_between(F.to_date("anio_mes", "yyyy-MM"), F.to_date("mes_cohorte", "yyyy-MM")).cast("int")`.
3. Clientes distintos por `mes_cohorte` y `meses_desde`, con un `pivot` sobre `meses_desde` (valores 0 a 5). Ojo: las columnas del pivot se llaman `"0"`, `"1"`... (texto).
4. Divide cada columna por la columna `"0"`.

In [0]:
pedidos_validos = pedidos_enriquecidos.filter(
    (F.col("estado") == "entregado") & (F.col("plan").isNotNull())
)

ventana_cliente = Window.partitionBy("id_cliente")

con_cohorte = pedidos_validos.withColumn(
    "mes_cohorte", 
    F.min("anio_mes").over(ventana_cliente)
)

con_meses_desde = con_cohorte.withColumn(
    "meses_desde",
    F.months_between(
        F.to_date(F.col("anio_mes"), "yyyy-MM"),
        F.to_date(F.col("mes_cohorte"), "yyyy-MM")
    ).cast("int")
)

actividad_cohortes = con_meses_desde.select(
    "mes_cohorte",
    "id_cliente",
    "meses_desde"
).filter(
    (F.col("meses_desde") >= 0) & (F.col("meses_desde") <= 5)
).distinct()

matriz_clientes = actividad_cohortes.groupBy("mes_cohorte") \
    .pivot("meses_desde", [0, 1, 2, 3, 4, 5]) \
    .agg(F.countDistinct("id_cliente"))

matriz_clientes = matriz_clientes.withColumnRenamed("0", "clientes_m0")
c0 = F.col("clientes_m0")

retencion = matriz_clientes.select(
    "mes_cohorte",
    F.col("clientes_m0").alias("clientes"),
    F.lit(100.0).alias("m0"),
    F.round(F.coalesce(F.col("1") / c0 * 100, F.lit(None)), 1).alias("m1"),
    F.round(F.coalesce(F.col("2") / c0 * 100, F.lit(None)), 1).alias("m2"),
    F.round(F.coalesce(F.col("3") / c0 * 100, F.lit(None)), 1).alias("m3"),
    F.round(F.coalesce(F.col("4") / c0 * 100, F.lit(None)), 1).alias("m4"),
    F.round(F.coalesce(F.col("5") / c0 * 100, F.lit(None)), 1).alias("m5")
).orderBy("mes_cohorte")
display(retencion)

mes_cohorte,clientes,m0,m1,m2,m3,m4,m5
2025-01,375,100.0,69.6,62.9,55.2,48.5,44.8
2025-02,246,100.0,68.7,63.4,54.1,51.2,null
2025-03,219,100.0,71.7,57.5,52.5,null,null
2025-04,235,100.0,71.5,59.6,null,null,null
2025-05,176,100.0,71.6,null,null,null,null
2025-06,133,100.0,null,null,null,null,null


In [0]:
# comprobación
filas = retencion.collect()
assert retencion.columns == ["mes_cohorte", "clientes", "m0", "m1", "m2", "m3", "m4", "m5"], f"Columnas: {retencion.columns}"
assert [r["mes_cohorte"] for r in filas] == ["2025-01", "2025-02", "2025-03", "2025-04", "2025-05", "2025-06"]
assert all(r["m0"] == 100.0 for r in filas), "m0 debe ser 100 en todas las cohortes"
registrados = pedidos_enriquecidos.filter("estado = 'entregado' AND plan IS NOT NULL").select("id_cliente").distinct().count()
assert sum(r["clientes"] for r in filas) == registrados, "Cada cliente debe pertenecer a una sola cohorte"
assert filas[-1]["m1"] is None and filas[0]["m5"] is not None
assert all(0 < r["m1"] < 100 for r in filas[:-1])
ok("Desafío A listo")

Desafío A listo


### Desafío B: segmentación de clientes
Marketing quiere una tabla para sus campañas. Crea `perfil_clientes` con una fila por cliente registrado que tenga al menos un pedido entregado, y estas columnas en este orden:

| columna | cálculo |
|---|---|
| `id_cliente` | |
| `n_pedidos` | pedidos entregados |
| `gasto_total` | suma de `total`, redondeada a 2 |
| `ultima_compra` | `fecha` del último pedido entregado |
| `dias_sin_comprar` | días entre `ultima_compra` y `FECHA_CORTE` (30 de junio de 2025) |
| `cuartil_gasto` | `ntile(4)` por `gasto_total` de mayor a menor |
| `segmento` | ver reglas abajo |

Reglas de `segmento`, en este orden:
- `"VIP activo"`: cuartil 1 y `dias_sin_comprar` <= 30
- `"VIP en riesgo"`: cuartil 1 y más de 30 días sin comprar
- `"Dormido"`: más de 60 días sin comprar
- `"Regular"`: el resto

Después:
1. Guárdalo como la tabla `gold_segmentos_clientes` y léela en `gold_segmentos`.
2. Crea `vip_en_riesgo` para que marketing los llame: los clientes del segmento `"VIP en riesgo"` con `id_cliente`, `nombre`, `email`, `telefono` (de `silver_clientes`), `gasto_total` y `dias_sin_comprar`, ordenados de mayor a menor gasto.

In [0]:
from datetime import date
FECHA_CORTE = date(2025, 6, 30)

clientes_base = pedidos_enriquecidos \
    .filter((F.col("estado") == "entregado") & (F.col("plan").isNotNull())) \
    .groupBy("id_cliente") \
    .agg(
        F.count("id_pedido").alias("n_pedidos"),
        F.round(F.sum("total"), 2).alias("gasto_total"),
        F.max("fecha").alias("ultima_compra")
    )
ventana_cuartil = Window.orderBy(F.col("gasto_total").desc())

perfil_clientes = clientes_base \
    .withColumn("dias_sin_comprar", F.datediff(F.lit(FECHA_CORTE), F.col("ultima_compra"))) \
    .withColumn("cuartil_gasto", F.ntile(4).over(ventana_cuartil))
perfil_clientes = perfil_clientes.withColumn(
    "segmento",
    F.when((F.col("cuartil_gasto") == 1) & (F.col("dias_sin_comprar") <= 30), "VIP activo")
     .when((F.col("cuartil_gasto") == 1) & (F.col("dias_sin_comprar") > 30), "VIP en riesgo")
     .when(F.col("dias_sin_comprar") > 60, "Dormido")
     .otherwise("Regular")
)
perfil_clientes.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_segmentos_clientes")

gold_segmentos = spark.table("gold_segmentos_clientes")
vip_en_riesgo = gold_segmentos \
    .filter(F.col("segmento") == "VIP en riesgo") \
    .join(silver_clientes, "id_cliente", "inner") \
    .select(
        "id_cliente",
        "nombre",
        "email",
        "telefono",
        "gasto_total",
        "dias_sin_comprar"
    ) \
    .orderBy(F.col("gasto_total").desc())

display(gold_segmentos.groupBy("segmento").agg(F.count("*").alias("n_clientes"), F.round(F.avg("gasto_total"), 2).alias("gasto_medio")))
display(vip_en_riesgo)

segmento,n_clientes,gasto_medio
VIP activo,289,1175.63
VIP en riesgo,57,1059.51
Regular,685,311.65
Dormido,353,217.01


id_cliente,nombre,email,telefono,gasto_total,dias_sin_comprar
275,Fiorella Gutiérrez,fiorella.gutierrez275@hotmail.com,911034023,3227.69,36
1091,Renzo Ramírez,null,943157933,2079.41,95
805,Milagros Gutiérrez,milagros.gutierrez805@yahoo.es,938372434,1857.44,61
731,Carlos Huamán,carlos.huaman731@yahoo.es,906703559,1803.79,61
88,Rosa Mamani,rosa.mamani88@yahoo.es,943425682,1671.16,33
640,César Flores,cesar.flores640@hotmail.com,926274294,1637.87,46
747,Ximena García,ximena.garcia747@yahoo.es,977286657,1629.98,40
1169,Rosa Torres,rosa.torres1169@gmail.com,null,1296.01,73
83,Bruno Quispe,bruno.quispe83@gmail.com,972094129,1269.3,36
505,Andrés Chávez,andres.chavez505@outlook.com,976831570,1253.61,31


In [0]:
# comprobación
assert gold_segmentos.columns == ["id_cliente", "n_pedidos", "gasto_total", "ultima_compra", "dias_sin_comprar", "cuartil_gasto", "segmento"], \
    f"Columnas: {gold_segmentos.columns}"
registrados = pedidos_enriquecidos.filter("estado = 'entregado' AND plan IS NOT NULL").select("id_cliente").distinct().count()
assert gold_segmentos.count() == registrados
segmentos = {r["segmento"] for r in gold_segmentos.select("segmento").distinct().collect()}
assert segmentos == {"VIP activo", "VIP en riesgo", "Dormido", "Regular"}, f"Segmentos: {segmentos}"
assert gold_segmentos.filter(F.col("segmento").startswith("VIP")).count() == gold_segmentos.filter("cuartil_gasto = 1").count(), \
    "Todos los del cuartil 1 son VIP (activos o en riesgo)"
assert gold_segmentos.filter((F.col("segmento") == "Dormido") & (F.col("dias_sin_comprar") <= 60)).count() == 0
assert vip_en_riesgo.columns == ["id_cliente", "nombre", "email", "telefono", "gasto_total", "dias_sin_comprar"]
assert vip_en_riesgo.count() == gold_segmentos.filter("segmento = 'VIP en riesgo'").count()
ok("Desafío B listo. Terminaste la práctica 2")

Desafío B listo. Terminaste la práctica 2


## Limpieza (opcional)
Para borrar todo lo que creó esta práctica (tablas, volumen y archivos), descomenta la línea y ejecútala:

In [0]:
# spark.sql(f"DROP SCHEMA IF EXISTS {CATALOGO}.{ESQUEMA} CASCADE")

## Para seguir
- Funciones de PySpark (la lista completa, incluidas las de arrays y fechas): https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html
- Delta Lake en Databricks (MERGE, time travel, historial): https://docs.databricks.com/aws/en/delta/
- En este notebook leímos todos los archivos cada vez. Cuando los archivos llegan continuamente, Databricks usa Auto Loader, que procesa solo los archivos nuevos: https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/
- El pipeline bronze, silver y gold que armamos a mano es justo lo que automatizan los pipelines declarativos de Lakeflow: https://docs.databricks.com/aws/en/dlt/